<a href="https://colab.research.google.com/github/psmail147/Mathematics-Repo/blob/master/DOCBOT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<font size="6"><h1><center><u><b>DocBot: Evaluating a Generative Chatbot Model to Assist in Giving Readable and Relevant Medical Advice.</b></u></center></h1>

<center>Philip Smith

Student Number: p2585980</center>

In this notebook we aim to alter the given chatbot base code which is a non domain specific generative chatbot into a domain specific chatbot (in this case healthcare and medical diagnosis). We want the chatbot (hencefoth called DocBot) to potentially diagnose, but also offer adivce on more pyschological issues, such as mental illness and metal health.

Goals:

1) Can we get the DocBot to output answers to medical questions that are a) legible and intelligible, and b) can the DocBot produce relevant answers to the input queires. For instance if a users query describes the symptoms of a common cold, will the DocBot either direct the user to a legitimate medical source (a general practionaer, or a medical website), or alternativly will the DocBot correctly diagnose the condition and or offer useful advice regarding treatment options.

The notebook is organised as follows: In section 1 we will clean the datasets to prepare them for the neural network models. Section 2 to section 4 demonstrates the Vocabularly and Vectorizer classes, the neural network architecture and the training routine. Section 5 to section 7 shows the evaluation and testing of the DocBot and section 8 to section 10 present a conclusion and future reccomendations. The first code block lists the libraries and packages used.








<h2><b>Contents</b></h2>

<ol>
  <li>DataSet Preprocessing
  <ul>
      <li>Dataset 1 (symptom query, diagnosis response)</li>
      <li>Dataset 2 (71 patient therapist conversations)</li>
      <li>Dataset 3 (Simulated Patient-Physician Medical Interviews)</li>
      <li>Dataset 4 (Diagnose Me)</li>
      <li>Creating the final Dataset</li>
  </ul>
  <li>Vocabularly and Vectorizer Classes</li>
  <li>Nueral Network Models</li>
  <li>Training</li>
  <li>Evaluation</li>
  <li>Quantitative Evaluation</li>
  <ul>
      <li>Bleu</li>
      <li>Cosine Test 1</li>
      <li>Cosine Test 2</li>
      <li>Readability Test</li>
  </ul>
  <li>Qaulitative Evaluation</li>
   <ul>
      <li>Phase 1: NHS Test Set</li>
      <li>Phase 2: Unsturctured Interaction</li>
      <li>Phase 3: Readability</li>
  </ul>
  <li>Conclusion</li>
  <li>Future Recommendations</li>
  <li>References</li>
</ol>


***Package and Library Installs.***

In [ ]:
!pip install sentence-transformers
!pip install tabulate
!pip install py-readability-metrics
!pip install sentence_transformers
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('bert-base-nli-mean-tokens')
from sentence_transformers import SentenceTransformer, util
model1 = SentenceTransformer('all-MiniLM-L6-v2')

from __future__ import absolute_import
from __future__ import division
from __future__ import print_function
from __future__ import unicode_literals

import torch
from torch.jit import script, trace
import torch.nn as nn
from torch import optim
import torch.nn.functional as F
import csv
import random
import re
import os
import unicodedata
import codecs
from io import open
import itertools
import math
import json
from pathlib import Path
from torchtext.data.metrics import bleu_score
import nltk
nltk.download('punkt')
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction
from nltk.corpus import stopwords
from tabulate import tabulate
from readability import Readability
import textwrap

USE_CUDA = torch.cuda.is_available()
device = torch.device("cuda" if USE_CUDA else "cpu")
print(device)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## **1. Dataset Preprocessing**

For most of the pre-processing dataset section, we use "ds" to mean "dataset", ds1 = dataset1, ds2 = dataset2... and so on. We note that medical datasets are not easy to come by for many reasons, not least because a lot of this data can be very private in nature and is monetizable for black markets where criminals obtain and sell private medical information. As such, medical data is subject to a lot of government regulations [3]. We had to rely on a lot of simulated medical conversations as a result, but these conversations were informed and created by medical professionals. We found four appropriate datasets for our purposes. The datasets are quite different from one another and we hope this will impart some diversity into the system. We will clean each dataset then merge them into one list to create a vocabulary and tensors for the neural network model.




<i>**Print function to monitor our datasets as they are changed in prepreation for a nuerual network model.**

In [ ]:
# CODE COPIED FROM Matthew Inkawhich.
corpus_name = "NLP"
corpus = os.path.join("drive/MyDrive/Colab Notebooks", corpus_name)

def printLines(file, n=10):
    with open(file, 'rb') as datafile: # rb appends line terminator (\n) as well as the "b" at the beggining of each line
        lines = datafile.readlines()
    for line in lines[:n]:
        print(line)

#printLines(os.path.join(corpus, "dataset.csv"))

### **Dataset 1 (symptom query, diagnosis response)**

The first data set is a small cvs file containing 41 unique diseases and multiple symptom combinations. Thus different input queries could map to the same illness<sup>[1]</sup>. This is not a conversational dataset. We initially trained the model with this dataset alone, achieving fast converge with a low average loss. The model was able to return the correct illness for a precise set of symptoms in the format that the csv file is structured as. But of course, this is very limited, and the model behaved more like a classification tool that a conversational chatbot.

<i>**View initial dataset**</i>

Note that the illness is in the first column and the symptoms are in the remaining columns to the left. We will reverse this later as the symptoms ought to be part of the query input. As can be seen below, there are inconsistent spaces between the diagnosis and symptoms and lots of commas to delete.

In [ ]:
printLines(os.path.join(corpus, "dataset.csv"))

b'Fungal infection,itching, skin_rash, nodal_skin_eruptions, dischromic _patches,,,,,,,,,,,,,\r\n'
b'Fungal infection, skin_rash, nodal_skin_eruptions, dischromic _patches,,,,,,,,,,,,,,\r\n'
b'Fungal infection,itching, nodal_skin_eruptions, dischromic _patches,,,,,,,,,,,,,,\r\n'
b'Fungal infection,itching, skin_rash, dischromic _patches,,,,,,,,,,,,,,\r\n'
b'Fungal infection,itching, skin_rash, nodal_skin_eruptions,,,,,,,,,,,,,,\r\n'
b'Fungal infection, skin_rash, nodal_skin_eruptions, dischromic _patches,,,,,,,,,,,,,,\r\n'
b'Fungal infection,itching, nodal_skin_eruptions, dischromic _patches,,,,,,,,,,,,,,\r\n'
b'Fungal infection,itching, skin_rash, dischromic _patches,,,,,,,,,,,,,,\r\n'
b'Fungal infection,itching, skin_rash, nodal_skin_eruptions,,,,,,,,,,,,,,\r\n'
b'Fungal infection,itching, skin_rash, nodal_skin_eruptions, dischromic _patches,,,,,,,,,,,,,\r\n'


<i>**Insert space after each diagnosis.**</i>

We created the insertspace() function below, it simply takes the old file, inserts a space after each comma in every line and writes the new output to a new file "ds1_space_insert_out".

In [ ]:
ds1_original = os.path.join(corpus, "dataset.csv")
ds1_space_insert_out = os.path.join(corpus, "ds1_space_insert.csv")

# insertspace function will insert a space after each comma in csv file.

def insertspace(old_file, new_file):
    with open(old_file,"r") as ofile:
        output = [", ".join(space.split(',')) for space in ofile.readlines()]
    with open(new_file,'w') as nfile:
        nfile.writelines(output)

insertspace(ds1_original ,ds1_space_insert_out)

# Print new csv file to check a space has been inserted after each comma.
printLines(os.path.join(corpus, "ds1_space_insert.csv"))

b'Fungal infection, itching,  skin_rash,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , \n'
b'Fungal infection,  skin_rash,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection, itching,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection, itching,  skin_rash,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection, itching,  skin_rash,  nodal_skin_eruptions, , , , , , , , , , , , , , \n'
b'Fungal infection,  skin_rash,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection, itching,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection, itching,  skin_rash,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection, itching,  skin_rash,  nodal_skin_eruptions, , , , , , , , , , , , , , \n'
b'Fungal infection, itching,  skin_rash,  nodal_skin_eruptions,  dischromic _patches, , , 

<i>**First comma delimiter replacement.**</i>

Replace first comma with delimeter \t to identify where the query and the repsonse are sepperated. Later we shall switch the elements of the pairs so the query symptoms are in the query position and the mapped diagnosis will be the response.

In [ ]:
ds1_space_insert_in = os.path.join(corpus, "ds1_space_insert.csv")
ds1_delimited_out = os.path.join(corpus, "ds1_delimited.csv")

def insert_t(old_file, new_file):
  with open(old_file) as ofile, open(new_file, "w") as nfile:
      for line in ofile:
          temp = line.split(',', 1)
          newline = '%s\t%s' % (','.join(temp[:1]), temp[1])
          nfile.write(newline)

insert_t(ds1_space_insert_in, ds1_delimited_out)

printLines(os.path.join(corpus, "ds1_delimited.csv"))

b'Fungal infection\t itching,  skin_rash,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , \n'
b'Fungal infection\t  skin_rash,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection\t itching,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection\t itching,  skin_rash,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection\t itching,  skin_rash,  nodal_skin_eruptions, , , , , , , , , , , , , , \n'
b'Fungal infection\t  skin_rash,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection\t itching,  nodal_skin_eruptions,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection\t itching,  skin_rash,  dischromic _patches, , , , , , , , , , , , , , \n'
b'Fungal infection\t itching,  skin_rash,  nodal_skin_eruptions, , , , , , , , , , , , , , \n'
b'Fungal infection\t itching,  skin_rash,  nodal_skin_eruptions,  dischromic _pat

<i>**Delete all commas.**</i>

We needed the commas for the previous step in order to insert the tab space after each disease label.

In [ ]:
in_text = os.path.join(corpus, "ds1_delimited.csv")
out_text = os.path.join(corpus, "ds1_final_pairs.txt")

with open(in_text) as old_file, open(out_text, "w") as new_file:
    for line in old_file:
      new_file.write(line.replace(",", ""))

printLines(os.path.join(corpus, "ds1_final_pairs.txt"))


b'Fungal infection\t itching  skin_rash  nodal_skin_eruptions  dischromic _patches             \n'
b'Fungal infection\t  skin_rash  nodal_skin_eruptions  dischromic _patches              \n'
b'Fungal infection\t itching  nodal_skin_eruptions  dischromic _patches              \n'
b'Fungal infection\t itching  skin_rash  dischromic _patches              \n'
b'Fungal infection\t itching  skin_rash  nodal_skin_eruptions              \n'
b'Fungal infection\t  skin_rash  nodal_skin_eruptions  dischromic _patches              \n'
b'Fungal infection\t itching  nodal_skin_eruptions  dischromic _patches              \n'
b'Fungal infection\t itching  skin_rash  dischromic _patches              \n'
b'Fungal infection\t itching  skin_rash  nodal_skin_eruptions              \n'
b'Fungal infection\t itching  skin_rash  nodal_skin_eruptions  dischromic _patches             \n'


### **Dataset 2 (71 patient therapist conversations)**

It should be noted that the transcribed part of this dataset was created using automated speech recognition (Google Cloud Speech-To-Text API ), thus not all words will be word for word perfect<sup>[2]</sup>. This dataset is conversational in nature containing psychiatric dialogues. There are 71 separate json files which will need merging into one file first.

<i>**Merge Converstaions**.</i>

The merge_json_conversations function takes 71 filenames and merges 71 json conversations into one json file for pre-processing.

In [ ]:
files = [f'c{index}.json' for index in range(1, 72)]
path_to_files = Path(r"drive/MyDrive/Colab Notebooks/NLP/PDC")
ds2_all_conversations_out = os.path.join(corpus, "ds2_all_conversations.json")

def merge_json_conversations(filenames):
    result = list()
    for file in filenames:
      file_path = path_to_files/file
      #print(file) # test to make sure we are actually inputting correct files.
      with open(file_path, 'r') as input_file:
            result.extend(json.load(input_file))

    with open(ds2_all_conversations_out, 'w') as output_file:
        json.dump(result, output_file)
        print(result)

merge_json_conversations(files)

[{'speaker': 1, 'dialogue': ['I am the psychiatrist here in this department.']}, {'speaker': 2, 'dialogue': ["I came to see you because my GP sent me to see you didn't he?"]}, {'speaker': 1, 'dialogue': ['Yeah, yeah.']}, {'speaker': 2, 'dialogue': ["He said come and see a trick cyclist cuz then you'll be all right."]}, {'speaker': 1, 'dialogue': ['Come and see us a trick cyclist a psychiatrist.']}, {'speaker': 2, 'dialogue': ['Yeah, he not he not had time to listen to me.', 'You see?', 'Okay.', "He's a GP.", "He's a doctor but I don't think he's very clever.", "He's not very clever because he's not okay getting to getting to this.", "You see won't let let me talk about this.", "He sort of listen to it sometimes but then sometimes he doesn't"]}, {'speaker': 1, 'dialogue': ['Right.', "Can I just clarify why you are here and why I'm here.", "I'm the psychiatrist."]}, {'speaker': 2, 'dialogue': ["Well, I've come to see you in the emergency clinic today."]}, {'speaker': 1, 'dialogue': ['Oka

<i>**Create Patient and Doctor Lines files**</I>

As can be seen from the print out above, this file conveniently has numerical key value, 1 or 2. 1 indicating a doctor line and 2 indicating patient line. We will utilise these keys to create our doctor patient lines.

In [ ]:
ds2_doctor_lines_out = os.path.join(corpus, "ds2_doctor_lines.txt")
ds2_patient_lines_out = os.path.join(corpus, "ds2_patient_lines.txt")

def extractjsondata(filename):
  with open(filename, 'r') as file:
    doctors = []
    patients = []
    data = json.load(file)
    for sentence in data:
      if (sentence["speaker"] == 1):
        doctors.append(sentence["dialogue"])
      else:
        patients.append(sentence["dialogue"])

  return doctors, patients

doctorlines, patientlines = extractjsondata(os.path.join(corpus, "ds2_all_conversations.json"))

with open(ds2_doctor_lines_out, 'w') as nfile:
  for sent in doctorlines:
    nfile.write("%s\n" % sent)

with open(ds2_patient_lines_out, 'w') as nfile:
  for sent in patientlines:
    nfile.write("%s\n" % sent)

Print doctor and patient lines.

In [ ]:
for line in patientlines[:10]:
  print(line)
print()
for line in doctorlines[:10]:
  print(line)

["I came to see you because my GP sent me to see you didn't he?"]
["He said come and see a trick cyclist cuz then you'll be all right."]
['Yeah, he not he not had time to listen to me.', 'You see?', 'Okay.', "He's a GP.", "He's a doctor but I don't think he's very clever.", "He's not very clever because he's not okay getting to getting to this.", "You see won't let let me talk about this.", "He sort of listen to it sometimes but then sometimes he doesn't"]
["Well, I've come to see you in the emergency clinic today."]
["So that's why I'm here.", 'Is that right?', 'Yeah, cuz I got to tell you about this.', "There's important stuff in there really important stuff.", 'Okay.', 'Well before we start with that I come to see you.']
['Yeah name name is a game.', "I don't bother with names.", "That's just too good, right?", 'Okay.', 'Call me Audrey.']
["How old is Odell you're only as old as the woman you feel aren't you?", 'I feel quite young and better than most women in their 20s.']
['My ID w

<i>**Merge patient and doctor files into one file**</i>

We alternate the lines to get sentence pairs, with delimiter between lines. Note, we have reversed the lines so the patient talks first (this will be reversed back to original order later when preparing the text file for the neural network model. This is simply because a separate dataset used for this application needed reversing, thus there is a chance we will have to change the order of query response pairs at the outset when massaging the datasets.



In [ ]:
t1 = os.path.join(corpus, "ds2_doctor_lines.txt")
t2 = os.path.join(corpus, "ds2_patient_lines.txt")
t3 = os.path.join(corpus, "ds2_doctor_patient_pairs.txt")

delimiter = "\t"

with open(t1) as text1, open(t2, 'r') as text2, open(t3, 'w') as text3:
  for line in text2:
    line1 = line.strip()
    line2 = text1.readline().strip()
    line3 = line1 + delimiter + line2
    text3.write(line3 + "\n")

printLines(os.path.join(corpus, "ds2_doctor_patient_pairs.txt"))

b'["I came to see you because my GP sent me to see you didn\'t he?"]\t[\'I am the psychiatrist here in this department.\']\n'
b'["He said come and see a trick cyclist cuz then you\'ll be all right."]\t[\'Yeah, yeah.\']\n'
b'[\'Yeah, he not he not had time to listen to me.\', \'You see?\', \'Okay.\', "He\'s a GP.", "He\'s a doctor but I don\'t think he\'s very clever.", "He\'s not very clever because he\'s not okay getting to getting to this.", "You see won\'t let let me talk about this.", "He sort of listen to it sometimes but then sometimes he doesn\'t"]\t[\'Come and see us a trick cyclist a psychiatrist.\']\n'
b'["Well, I\'ve come to see you in the emergency clinic today."]\t[\'Right.\', "Can I just clarify why you are here and why I\'m here.", "I\'m the psychiatrist."]\n'
b'["So that\'s why I\'m here.", \'Is that right?\', \'Yeah, cuz I got to tell you about this.\', "There\'s important stuff in there really important stuff.", \'Okay.\', \'Well before we start with that I come to se

<i>**Remove backslashes, quoatation marks and square brackets**

In [ ]:
ds2_pairs_in = os.path.join(corpus, "ds2_doctor_patient_pairs.txt")
ds2_final_pairs_out = os.path.join(corpus, "ds2_final_pairs.txt")

with open(ds2_pairs_in) as ofile, open(ds2_final_pairs_out, "w") as nfile:
    for line in ofile:
      nfile.write(line.replace('"', '').replace("'", "").replace("[", "").replace("]", ""))

printLines(os.path.join(corpus, "ds2_final_pairs.txt"))


b'I came to see you because my GP sent me to see you didnt he?\tI am the psychiatrist here in this department.\n'
b'He said come and see a trick cyclist cuz then youll be all right.\tYeah, yeah.\n'
b'Yeah, he not he not had time to listen to me., You see?, Okay., Hes a GP., Hes a doctor but I dont think hes very clever., Hes not very clever because hes not okay getting to getting to this., You see wont let let me talk about this., He sort of listen to it sometimes but then sometimes he doesnt\tCome and see us a trick cyclist a psychiatrist.\n'
b'Well, Ive come to see you in the emergency clinic today.\tRight., Can I just clarify why you are here and why Im here., Im the psychiatrist.\n'
b'So thats why Im here., Is that right?, Yeah, cuz I got to tell you about this., Theres important stuff in there really important stuff., Okay., Well before we start with that I come to see you.\tOkay, and your GP has asked me to see you.\n'
b'Yeah name name is a game., I dont bother with names., Thats

### **Dataset 3:Simulated Patient-Physician Medical Interviews**


This dataset comprises 272 medical cases between patient and doctor. These are transcribed simulated medical conversations. The conversations were created with the aid of medical professionals and medical students. The transcriptions came from audio conversations and were transcribed manually. Again, we acknowledge that there may be errors in the dataset. Most of the medical queries are related to respiratory conditions, as well as musculoskeletal (16.9%), gastrointestinal (2.2%), cardiac (1.8%) and dermatological (0.4%)<sup>[3][4]</sup>.



<i>**Merge 272 files**</i>

We will save the seperate medical conversations into a list and unload that list into a txt file in the following two code blocks.

In [ ]:
files = [f's{i}.txt' for i in range(1, 273)]
path_to_files = Path(r"drive/MyDrive/Colab Notebooks/NLP/PDC1")
new_file = os.path.join(corpus, "ds3_merged_files.txt")

def merge_text_conversations(filenames):
    result = list()
    for file in filenames:
      file_path = path_to_files/file
      #print(file) # test to make sure we are actually inputting correct files.
      with open(file_path, 'rb') as input_file:
            #result.extend(json.load(input_file))
            result.extend(input_file)
    #with open(ds3_merged_files_out, 'w') as output_file:
        #json.dump(result, output_file)
    return result

merged_files = merge_text_conversations(files)

In [ ]:
with open(new_file, 'wb') as ofile:
  for item in merged_files:
    ofile.write(item)

printLines(new_file)

b'D: What brought you in today?\n'
b'\n'
b"P: Sure, I'm I'm just having a lot of chest pain and and so I thought I should get it checked out.\n"
b'\n'
b'D: OK, before we start, could you remind me of your gender and age? \n'
b'\n'
b"P: Sure 39, I'm a male.\n"
b'\n'
b'D: OK, and so when did this chest pain start?\n'
b'\n'


This dataset contains empty lines, we first want to  remove them. We had to manually remove a small portion of lines where all characters were sepaerted by spaces.

In [ ]:
#GET RID OF EMPTY LINES
old_file = os.path.join(corpus, "ds3_merged_files.txt")
new_file = os.path.join(corpus, "ds3_lines_removed_test.txt") # we had to manually remove some lines
#if we run this code again, it will create a problematic file where some lines see spaces between all characters
# we have manually removed these lines for this demonstration

with open(old_file,'rb') as ofile:
  with open(new_file, 'wb') as nfile:
    for line in ofile:
        if not line.isspace():
            nfile.write(line)
printLines(new_file)

b'D: What brought you in today?\n'
b"P: Sure, I'm I'm just having a lot of chest pain and and so I thought I should get it checked out.\n"
b'D: OK, before we start, could you remind me of your gender and age? \n'
b"P: Sure 39, I'm a male.\n"
b'D: OK, and so when did this chest pain start?\n'
b"P: It started last night, but it's becoming sharper.\n"
b'D: OK, and where is this pain located? \n'
b"P: It's located on the left side of my chest.\n"
b'D: OK, and, so how long has it been going on for then if it started last night?\n'
b'P: So I guess it would be a couple of hours now, maybe like 8.\n'


<i>**Create patient and doctor lines**</i>

We have used the "D" and "P" characters at the start of each line to identify doctors and patients lines, we need to remove them after using them.

In [ ]:
new_file_temp = os.path.join(corpus, "ds3_lines_removed.txt")

def getDoctorAndPatientLines(filename):
  with open(filename, 'r') as file:
    doctors = []
    patients = []
    #data = file.read
    for sentence in file:
      if (sentence[0] == "D"):
        doctors.append(sentence)
      elif (sentence[0] == "P"):
        patients.append(sentence)


  return doctors, patients

docs, pats = getDoctorAndPatientLines(new_file_temp)

# now delete "D" and "P" from start of each line.
doc_out_file = os.path.join(corpus, "ds3_doctor_lines.txt")
pat_out_file = os.path.join(corpus, "ds3_patient_lines.txt")

with open(doc_out_file, 'w') as output_file:
  for line in docs:
    #for line in input_file:
      output_file.write(line.strip("D:"))

with open(pat_out_file, 'w') as output_file:
  for line in pats:
    #for line in input_file:
      output_file.write(line.strip("P:"))

In [ ]:
printLines(pat_out_file)

b" Sure, I'm I'm just having a lot of chest pain and and so I thought I should get it checked out.\n"
b" Sure 39, I'm a male.\n"
b" It started last night, but it's becoming sharper.\n"
b" It's located on the left side of my chest.\n"
b' So I guess it would be a couple of hours now, maybe like 8.\n'
b" I would say it's been pretty constant, yeah.\n"
b" I'd say it's pretty sharp, yeah.\n"
b' Um not laying down helps.\n'
b' Yes, definitely.\n'
b' No.\n'


<i>**Merge all doctor patient conversations into one file.**</i>

Later these will be joined with other datasets and the lines will be shuffled. Again, the pairs will be reversed to original order later.

In [ ]:
f1 = os.path.join(corpus, "ds3_doctor_lines.txt")
f2 = os.path.join(corpus, "ds3_patient_lines.txt")
f3 = os.path.join(corpus, "ds3_final_pairs.txt")

delimiter = "\t"

with open(f1) as text1, open(f2, 'r') as text2, open(f3, 'w') as text3:
  for line in text2:
    line1 = line.strip()
    line2 = text1.readline().strip()
    line3 = line1 + delimiter + line2
    text3.write(line3 + "\n")

printLines(os.path.join(corpus, "ds3_final_pairs.txt"))

b"Sure, I'm I'm just having a lot of chest pain and and so I thought I should get it checked out.\tWhat brought you in today?\n"
b"Sure 39, I'm a male.\tOK, before we start, could you remind me of your gender and age?\n"
b"It started last night, but it's becoming sharper.\tOK, and so when did this chest pain start?\n"
b"It's located on the left side of my chest.\tOK, and where is this pain located?\n"
b'So I guess it would be a couple of hours now, maybe like 8.\tOK, and, so how long has it been going on for then if it started last night?\n'
b"I would say it's been pretty constant, yeah.\tOK. Has it been constant throughout that time, or uh, or changing?\n"
b"I'd say it's pretty sharp, yeah.\tOK, and how would you describe the pain? People will use words sometimes like sharp, burning, achy.\n"
b"Um not laying down helps.\tSharp OK. Uh, anything that you have done tried since last night that's made the pain better?\n"
b'Yes, definitely.\tOK, so do you find laying down makes the pain wor

### **Dataset 4: "Diagnose Me"**

This dataset is intended to give the DocBot its main conversational ability. There are many large datasets available for this task, such as the movie_corpus and endless Twitter, Reddit and other social media datasets. The issues we have with these is that they will contain a lot of slang words and phrases, colloquialisms and profanity. We are trying to create a chatbot that can converse like a health care professional, we certainly don't want it swearing at the patients.

To this end, dataset 4 is our conversational dataset and it is very large. It contains 257468 different conversation pairs (patient query, doctor response)<sup>[5]</sup>. Initially we were eager to use the entire dataset, but it dominates the other datasets when we merge them. The pre-processing code that follows extracts a more manageable portion of the dataset. However, as can be seen from the evaluation part of this notebook, these lines from this dataset are very long and we suspect this made training very challenging in terms of time complexity, long term memory dependancy and resulted in a model that could not handle short input queries.


<i>**View the initial json dataset**</i>

As can been seen from the output, each conversation has an integer id starting at 0. The task is get to the "Patient" and "Doctor" keys in order to extract the values (the sentences).

In [ ]:
printLines(os.path.join(corpus, "diagnose_me_original_dataset.json"))

b'[\n'
b'    {\n'
b'        "id":0,\n'
b'        "Description":"Q. What does abutment of the nerve root mean?",\n'
b'        "Doctor":"Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online --> https:\\/\\/www.icliniq.com\\/ask-a-doctor-online\\/neurologist  ",\n'
b'        "Patient":"Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\\u00a0annular bulging and tear?"\n'
b'    },\n'
b'    {\n'
b'        "id":1,\n'
b'        "Description":"Q. Every time I eat spicy food, I poop blood. Why?",\n'


<i>**Extract "Doctor" and "Patient" sentences.**</i>

In [ ]:
original_dataset = os.path.join(corpus, "diagnose_me_original_dataset.json")
doctorlines = []
patientlines = []
with open(original_dataset) as jsonFile:
    jsonObject = json.load(jsonFile)
    for line in jsonObject[:75000]:
      doctorlines.append(line['Doctor'])
      patientlines.append(line['Patient'])



We will take a quick look at the resulting lists to make sure they contain the information we want.

In [ ]:
for line in patientlines[:11]: # change patientlines to "doctorlines" to view the doctor responses
  print(line)


Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for annular bulging and tear?
Hi doctor, I am a 26 year old male. I am 5 feet and 9 inches tall and weigh 255 pounds. When I eat spicy food, I poop blood. Sometimes when I have constipation as well, I poop a little bit of blood. I am really scared that I have colon cancer. I do have diarrhea often. I do not have a family history of colon cancer. I got blood tests done last night. Please find my reports attached.
Hello doctor, I am 48 years old. I am experiencing weak erection and difficulty in sustaining the same. This condition was observed 10 years back. Also, there is premature ejaculation. Other physical ailments that I have are, I am suffering from hypertension and taking Amlopres-L (Amlodipine and Lisinopril) for the last 10 years, high cholesterol and triglycerides. My cholesterol level is 225 and triglyceride is 200 for the last 12 year

Now we shall write these lists into text files ready for merging into sentence pairs

In [ ]:
newfile2 = os.path.join(corpus, "ds4_doctor_lines.txt")
newfile3 = os.path.join(corpus, "ds4_patient_lines.txt")

with open(newfile2, 'w') as nfile:
  for sent in doctorlines:
    nfile.write("%s\n" % sent)

with open(newfile3, 'w') as nfile:
  for sent in patientlines:
    nfile.write("%s\n" % sent)

In [ ]:
# print previously created files
printLines(newfile3)

b'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xc2\xa0annular bulging and tear?\n'
b'Hi doctor, I am a 26 year old male. I am 5 feet and 9 inches tall and weigh 255 pounds. When I eat spicy\xc2\xa0food, I poop blood. Sometimes when I have constipation as well, I poop a little bit of blood. I am really scared that I have colon cancer. I do have diarrhea often. I\xc2\xa0do not have a\xc2\xa0family history of colon\xc2\xa0cancer. I got blood tests done last night. Please find my reports attached.\n'
b'Hello doctor, I am 48 years old. I am experiencing weak erection and difficulty in sustaining the same. This condition was observed 10 years back. Also, there is premature ejaculation. Other physical ailments that I have are, I am suffering from hypertension and taking Amlopres-L (Amlodipine and Lisinopril) for the last 10 years, high cholesterol and triglycerides. My cholesterol level is 2

Now we shall merge these two files into one, ready query response pairs.

In [ ]:
in1 = os.path.join(corpus, "ds4_doctor_lines.txt")
in2 = os.path.join(corpus, "ds4_patient_lines.txt")
out1 = os.path.join(corpus, "ds4_patient_doctor_lines.txt")

delimiter = "\t"

with open(in1) as text1, open(in2, 'r') as text2, open(out1, 'w') as text3:
  for line in text2:
    line1 = line.strip()
    line2 = text1.readline().strip()
    line3 = line2 + delimiter + line1
    text3.write(line3 + "\n")

printLines(os.path.join(corpus, "ds4_patient_doctor_lines.txt"))

b'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online --> https://www.icliniq.com/ask-a-doctor-online/neurologist\tHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xc2\xa0annular bulging and tear?\n'
b'Hello. I have gone through your information and test reports (attachment removed to protect patient identity). So, in view of that, there are a couple of things that I can opine upon: Hope that helps. For more information consult a general surgeon online --> https://icliniq.com./ask-a-doctor-online/general-surgeon\tHi doctor, I am a 26 year old male. I am 5 feet and 9 inches tall and weigh 255 pounds. When I eat spicy\xc2\xa0food, I poop blood. Sometimes when I have constipation as well, I poop a little bit of blood. I am really scared that I have colon cancer. I do have diarrhea often.

<i>**Final cleaning up of the data.**</i>

We remove brackets and apostrophes. We left the URL's in as the DocBot was able to redirect users to these websites in some of its responses, we found this to be a desirable response in lieu of giving any actual medical advice.

In [ ]:
in_file = os.path.join(corpus, "ds4_patient_doctor_lines.txt")
out_file = os.path.join(corpus, "ds4_final_pairs.txt")

with open(in_file) as ofile, open(out_file, "w") as nfile:
    for line in ofile:
      nfile.write(line.replace('"', '').replace("'", "").replace("(", "").replace(")", ""))

# re.sub(r'http\S+', '', my_string) # for url deletion later.

printLines(os.path.join(corpus, "ds4_final_pairs.txt"))

b'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online --> https://www.icliniq.com/ask-a-doctor-online/neurologist\tHi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xc2\xa0annular bulging and tear?\n'
b'Hello. I have gone through your information and test reports attachment removed to protect patient identity. So, in view of that, there are a couple of things that I can opine upon: Hope that helps. For more information consult a general surgeon online --> https://icliniq.com./ask-a-doctor-online/general-surgeon\tHi doctor, I am a 26 year old male. I am 5 feet and 9 inches tall and weigh 255 pounds. When I eat spicy\xc2\xa0food, I poop blood. Sometimes when I have constipation as well, I poop a little bit of blood. I am really scared that I have colon cancer. I do have diarrhea often. I

# **Creating The Final Dataset**

We now take all final text files for the four prepared datasets and merge them into one dataset. After this we will shuffle the lines as we want to remove the risk that a batch of query response pairs may be just from one single source.

<i>**Merge all four datasets into one file.**</i>

In [ ]:
dataset_model_ready_out = os.path.join(corpus, "merged_dataset.txt")
inputfile1 = os.path.join(corpus, "ds1_final_pairs.txt")
inputfile2 = os.path.join(corpus, "ds2_final_pairs.txt")
inputfile3 = os.path.join(corpus, "ds3_final_pairs.txt")
inputfile4 = os.path.join(corpus, "ds4_final_pairs.txt")

def makeMergedDataSet(file1, file2, file3, file4, outputfile):
  with open(file1, 'r') as f1:
    data1 = f1.read()
  with open(file2, 'r') as f2:
    data2 = f2.read()
  with open(file3, 'r') as f3:
    data3 = f3.read()
  with open(file4, 'r') as f4:
    data4 = f4.read()
  data1 += data2
  data1 += data3
  data1 += data4
  with open (outputfile, 'w') as out_file:
    out_file.write(data1)

makeMergedDataSet(inputfile1, inputfile2, inputfile3, inputfile4, dataset_model_ready_out)

In [ ]:
printLines(os.path.join(corpus, "merged_dataset.txt"))

b'Fungal infection\t itching  skin_rash  nodal_skin_eruptions  dischromic _patches             \n'
b'Fungal infection\t  skin_rash  nodal_skin_eruptions  dischromic _patches              \n'
b'Fungal infection\t itching  nodal_skin_eruptions  dischromic _patches              \n'
b'Fungal infection\t itching  skin_rash  dischromic _patches              \n'
b'Fungal infection\t itching  skin_rash  nodal_skin_eruptions              \n'
b'Fungal infection\t  skin_rash  nodal_skin_eruptions  dischromic _patches              \n'
b'Fungal infection\t itching  nodal_skin_eruptions  dischromic _patches              \n'
b'Fungal infection\t itching  skin_rash  dischromic _patches              \n'
b'Fungal infection\t itching  skin_rash  nodal_skin_eruptions              \n'
b'Fungal infection\t itching  skin_rash  nodal_skin_eruptions  dischromic _patches             \n'


<i>**More data cleaning.**</i>

Delete quotation marks, they are creating odd strings such as "\xe2\x80\x98".

In [ ]:
in_file = os.path.join(corpus, "merged_dataset.txt")
out_file = os.path.join(corpus, "merged_dataset_clean.txt")

with open(in_file) as ofile, open(out_file, "w") as nfile:
    for line in ofile:
      nfile.write(line.replace("'", ''))



<i>**Shuffle Lines**</i>

In order that no batch is selected from the same dataset, we will shuffle the lines.

In [ ]:
outF = os.path.join(corpus, "dataset_shuffled.txt")
lines = open(os.path.join(corpus, "merged_dataset_clean.txt")).readlines()
random.shuffle(lines)
open(outF, 'w').writelines(lines)

In [ ]:
printLines(outF)

b'Dear Patient,   Thank you very much for your questions. There are different types of pathological heart beating conditions or arrhythmia. It is very hard for a physician without auscultation, physical examination and ECG to diagnose such  conditions. However, thanks to health care magic forum you have a golden opportunity to get an advice from a physician to direct you to a right person before worsening your symptoms. Anyway, according to your description about your condition, I feel that you have premature heart beats. Premature heat beats are subdivided into premature atrial contractionsPACs and premature ventricular contractionsPVCs.  Also, Atrial fibrillation itself can give an irregular rhythm. Some irregular heart rhythms can be life threatening.               Therefore, my advice is consult your family physician first. Explain him all your physical complaints like skipped heart rate, racing your heart,shortness of breathing, things which can trigger these conditions for exampl

<i>**Training and Testing Dataset Creation**</i>

Split into two files, one for testing during evaluation and one for training. We will save 10% for testing.

In [ ]:
num_lines = sum(1 for line in open(outF))
test_size = num_lines * 0.10 # we will extract 10% of the shuffled dataset for testing
test_size = int(test_size)

read_file = os.path.join(corpus, "dataset_shuffled.txt")
test_file = os.path.join(corpus, "test_set_test.txt")
train_file = os.path.join(corpus, "train_set_test.txt")


def makeTestAndTrainFiles(readF, testwrite, trainwrite):
  with open(readF, 'r') as f1:
    data = []
    for i in range(1, test_size):#we can start at the beggining as we have shuffled the lines
      data.append(f1.readline())
  with open(testwrite, 'w') as w1:
    for line in data:
      w1.write(line)

  # check for duplicates before creating training set.

  with open(readF, 'r') as f2:
    data2 = []
    data1 = f2.readlines()
    for i in range(test_size, num_lines):
      data2.append(data1[i])
    testSet = set(data)
    trainSet = set(data2)
    linesInBoth = testSet.intersection(trainSet)
    isEmpty = True if not linesInBoth else False
    if (not isEmpty): # we need to remove these lines from training list before writing to train file
        for line in linesInBoth:
          for line1 in data2:
            if line == line1:
              data2.remove(line1)

    with open(trainwrite, 'w') as w2:
      for line in data2:
        w2.write(line)

makeTestAndTrainFiles(read_file, test_file, train_file)

Double check to make sure there are no patient doctor pairs in both the train and test set.

In [ ]:
# testing for copies
testList = []
trainList = []
with open(test_file, 'r') as test:
  with open(train_file, 'r') as train:
    testList = test.readlines()
    trainList = train.readlines()

testset = set(testList)
trainset = set(trainList)
linesInBoth1 = testset.intersection(trainset)
isEmpty = True if not linesInBoth1 else False
print(isEmpty)
print(len(linesInBoth1))
print(sum(1 for line in open(test_file)))

file = open(train_file, "rt")
data = file.read()
words = data.split()

print('Number of words in text file :', len(words))



False
17
9704
Number of words in text file : 10914130


Our final training file contains 74368 conversation pairs.

In [ ]:
training_dataset = os.path.join(corpus, "train_set.txt")
print(sum(1 for line in open(training_dataset)))

74368


Here are the first 10 lines.

In [ ]:
printLines(train_file)

b'Every priest has to give a sermon every Parish Priest., Some people are really good at sermons and some people are really not good at them., Im both picky and kind of skeptical about things., I feel like one of sermons are good., So anyway, a lot of people end up talking about suffering as something that is a good thing or it sure sounds like theyre talking about suffering as a good thing., Maybe theyre not trying to say that but thats what they end up saying by accident., I dont really feel like this is done me any good.\tAbout how many is that?\n'
b'Yes, and so that was so all the sudden we got this completely upside down relationship.\tI can see that its painful for you., Can you tell me what youre feeling right now as youre telling me about it?\n'
b'Hi. I have read through your query in detail. Please find my observations below. The contrast CT scan pelvis has exposed you to radiation of 8 mSv as recorded on the machine. For a 36 year old female patient, this means you are expose

# **Vocabularly and Vectorizer Classes**

Most of the following classes required no alteration, we will briefly comment on the minor changes made.

<i>**Vocabulary Class**</i>

A get() method was added to the vocab class. We did not want the end user to notice when a word did not exist in the vocab, thus we had to overide the Attribute Error thrown from the original version of this program (see Evaluation section for more details).



In [ ]:
# Default word tokens
PAD_token = 0  # Used for padding short sentences
SOS_token = 1  # Start-of-sentence token
EOS_token = 2  # End-of-sentence token

class Voc:

      def __init__(self, name):
          self.name = name
          self.trimmed = False
          self.word2index = {}
          self.word2count = {}
          self.index2word = {PAD_token: "PAD", SOS_token: "SOS", EOS_token: "EOS"}
          self.num_words = 3  # Count SOS, EOS, PAD

      def addSentence(self, sentence):
          for word in sentence.split(' '):
              self.addWord(word)

      def addWord(self, word):
          if word not in self.word2index:
              self.word2index[word] = self.num_words
              self.word2count[word] = 1
              self.index2word[self.num_words] = word
              self.num_words += 1
          else: # if word already in voc, just update the count of that word.
              self.word2count[word] += 1

      # Remove words below a certain count threshold
      def trim(self, min_count):
          if self.trimmed:
              return
          self.trimmed = True

          keep_words = []

          for k, v in self.word2count.items():
              if v >= min_count:
                  keep_words.append(k)

          print('keep_words {} / {} = {:.4f}'.format(
              len(keep_words), len(self.word2index), len(keep_words) / len(self.word2index)
          ))

         # Reinitialize dictionaries
          self.word2index = {}
          self.word2count = {}
          self.index2word = {PAD_token: "PAD", SOS_token: "SOS", EOS_token: "EOS"}
          self.num_words = 3 # Count default tokens

          for word in keep_words:
              self.addWord(word)

      # check if word is in vocab.
      def get(self, key, default=None):
          return self.word2count.get(key, default)


We have commented out some of the code below when creating Vocabularies and pairs for training. We had to create different vocabularies depending on the MAX_LENGTH of the different versions of the DocBot. When we built our models for training and evaluation we parse in different MAX_LENGTHS to create the vocabularies and the pairs.

In [ ]:
training_dataset = os.path.join(corpus, "train_set.txt")

# Turn a Unicode string to plain ASCII, thanks to
# https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
    )

# Lowercase, trim, and remove non-letter characters
def normalizeString(s):
    s = unicodeToAscii(s.lower().strip())
    s = re.sub(r"([.!?])", r" \1", s)
    s = re.sub(r"[^a-zA-Z.!?]+", r" ", s)
    s = re.sub(r"\s+", r" ", s).strip()
    return s

# Read query/response pairs and return a voc object
def readVocs(datafile, corpus_name):
    print("Reading lines...")
    # Read the file and split into lines
    lines = open(datafile, encoding='utf-8').\
        read().strip().split('\n')
    # Split every line into pairs and normalize
    pairs = [[normalizeString(s) for s in l.split('\t')] for l in lines]
    voc = Voc(corpus_name)
    return voc, pairs #return two lists

# Returns True iff both sentences in a pair 'p' are under the MAX_LENGTH threshold
def filterPair(p, MAX_LENGTH):
    # Input sequences need to preserve the last word for EOS token
    return len(p[0].split(' ')) < MAX_LENGTH and len(p[1].split(' ')) < MAX_LENGTH

# Filter pairs using filterPair condition
def filterPairs(pairs, MAX_LENGTH):
    return [pair for pair in pairs if filterPair(pair, MAX_LENGTH)]

# Using the functions defined above, return a populated voc object and pairs list
def loadPrepareData(corpus, corpus_name, datafile, save_dir, MAX_LENGTH):
    print("Start preparing training data ...")
    voc, pairs = readVocs(datafile, corpus_name)
    print("Read {!s} sentence pairs".format(len(pairs))) # without curly braces, it would print literal "!s", with curly braces it uses it as a placeholder for number of pairs in paris, .len(pairs)
    pairs = filterPairs(pairs, MAX_LENGTH)
    print("Trimmed to {!s} sentence pairs".format(len(pairs)))
    print("Counting words...")
    for pair in pairs:
        voc.addSentence(pair[0])
        voc.addSentence(pair[1])
    print("Counted words:", voc.num_words)
    return voc, pairs


# Load/Assemble voc and pairs
#save_dir = os.path.join(corpus, "Model Save Folder")
#voc1, pairs1 = loadPrepareData(corpus, corpus_name, training_dataset, save_dir, 100)
#voc2, pairs2 = loadPrepareData(corpus, corpus_name, training_dataset, save_dir, 70)
# Print some pairs to validate

def swap(list):
  for pair in list:
    temp = pair[0]
    pair[0] = pair[1]
    pair[1] = temp

#swap(pairs1)
#swap(pairs2)

#print("\npairs:")
#for pair in pairs1[:10]:
 #   print(pair)
#for pair in pairs2[:10]:
 #   print(pair)

In [ ]:
MIN_COUNT = 1   # Minimum word count threshold for trimming

def trimRareWords(voc, pairs, MIN_COUNT):
    # Trim words used under the MIN_COUNT from the voc
    voc.trim(MIN_COUNT)
    # Filter out pairs with trimmed words
    keep_pairs = []
    for pair in pairs:
        input_sentence = pair[0]
        output_sentence = pair[1]
        keep_input = True
        keep_output = True
        # Check input sentence
        for word in input_sentence.split(' '):
            if word not in voc.word2index:
                keep_input = False
                break
        # Check output sentence
        for word in output_sentence.split(' '):
            if word not in voc.word2index:
                keep_output = False
                break

        # Only keep pairs that do not contain trimmed word(s) in their input or output sentence
        if keep_input and keep_output:
            keep_pairs.append(pair)

    print("Trimmed from {} pairs to {}, {:.4f} of total".format(len(pairs), len(keep_pairs), len(keep_pairs) / len(pairs)))
    return keep_pairs


# Trim voc and pairs
#pairs1 = trimRareWords(voc1, pairs1, MIN_COUNT)
#pairs2 = trimRareWords(voc2, pairs2, MIN_COUNT)

<i>**Create Tensors Ready for Neural Network Model**</i>

Commented out code below for this demonstration of the DocBot.

In [ ]:
def indexesFromSentence(voc, sentence):
    return [voc.word2index[word] for word in sentence.split(' ')] + [EOS_token]


def zeroPadding(l, fillvalue=PAD_token):
    return list(itertools.zip_longest(*l, fillvalue=fillvalue))

def binaryMatrix(l, value=PAD_token):
    m = []
    for i, seq in enumerate(l):
        m.append([])
        for token in seq:
            if token == PAD_token:
                m[i].append(0)
            else:
                m[i].append(1)
    return m

# Returns padded input sequence tensor and lengths
def inputVar(l, voc):
    indexes_batch = [indexesFromSentence(voc, sentence) for sentence in l]
    lengths = torch.tensor([len(indexes) for indexes in indexes_batch])
    padList = zeroPadding(indexes_batch)
    padVar = torch.LongTensor(padList)
    return padVar, lengths

# Returns padded target sequence tensor, padding mask, and max target length
def outputVar(l, voc):
    indexes_batch = [indexesFromSentence(voc, sentence) for sentence in l]
    max_target_len = max([len(indexes) for indexes in indexes_batch])
    padList = zeroPadding(indexes_batch)
    mask = binaryMatrix(padList)
    mask = torch.BoolTensor(mask)
    padVar = torch.LongTensor(padList)
    return padVar, mask, max_target_len

# Returns all items for a given batch of pairs
def batch2TrainData(voc, pair_batch):
    pair_batch.sort(key=lambda x: len(x[0].split(" ")), reverse=True)
    input_batch, output_batch = [], []
    for pair in pair_batch:
        input_batch.append(pair[0])
        output_batch.append(pair[1])
    inp, lengths = inputVar(input_batch, voc)
    output, mask, max_target_len = outputVar(output_batch, voc)
    return inp, lengths, output, mask, max_target_len


# Example for validation
small_batch_size = 5
#batches1 = batch2TrainData(voc1, [random.choice(pairs1) for _ in range(small_batch_size)])
#batches2 = batch2TrainData(voc2, [random.choice(pairs2) for _ in range(small_batch_size)])

#input_variable1, lengths1, target_variable1, mask1, max_target_len1 = batches1
#input_variable2, lengths2, target_variable2, mask2, max_target_len2 = batches2


#print("input_variable:", input_variable1)
#print("lengths:", lengths1)
#print("target_variable:", target_variable1)
#print("mask:", mask1)
#print("max_target_len:", max_target_len1)

# **Nueral Network Models**

We will be using the Recurrent Neural Network model created by Matthew Inkawhich to train our chatbot, the original code can be found in <sup>[6]</sup>.
Where the code is copied completely we will indicate this using comments. If a modification has been made by more than two or three lines of code, we shall indicate that modifications have been made by stating "#MODIFICATIONS MADE" at the beginning of the code block.

This model is referred to as a conditional language model which was first developed to help translate from one language to another. More broadly, and in this use case, it is used to generate a response sequence of varying length given an input query of varying length. The architecture is composed of two Recurrent Neural Networks (RNN), an encoder and a decoder. It is called a "conditional" language model because the decoder, rather than being conditioned by a zero vector as seen in more "vanilla" RNN's, it is conditoned by a context vector that is returned by the encoder RNN. Thus the decoder is conditioned with useful (hopefully) semantic and meaningful information from the input query sequence<sup>[14][15][6]</sup>.

We have altered the neural network by implementing a Long Short Term Memory unit (LSTM) RNN, whereby the user can choose between the original Gated Recurrent Unit RNN or the LSTM unit. We include an LSTM option as this is more commonly used in sequence-to-sequence modelling, especially where long term dependencies are important and given the length of sequences, we are processing we could not afford to not implement the LSTM. We note that the LSTM is less efficient due to having three update gates rather than two as seen in the GRU <sup>[16]</sup>. We noticed this when comparing the training time between the different models.


**Encoder**

Our Encoder gives the user the option to change the central computational unit from the Gated Recurrent Unit (GRU) to a Long Short Term Memory (LSTM) unit. A nn_type (neural network type) attribute was implemented in the EncoderRNN class along with conditional statement to activate whichever network was parsed to it when a model is trained or evaluated.

In [ ]:
# MODIFICATIONS MADE
class EncoderRNN(nn.Module):
    def __init__(self, hidden_size, embedding, nn_type, n_layers=1, dropout=0):
        super(EncoderRNN, self).__init__()
        self.n_layers = n_layers
        self.hidden_size = hidden_size
        self.embedding = embedding
        self.nn_type = nn_type

        # Initialize GRU; the input_size and hidden_size params are both set to 'hidden_size'
        #   because our input size is a word embedding with number of features == hidden_size
        if (self.nn_type == "GRU"):
          print("You are ready to train or evaluate with a GRU RNN.")
          self.gru = nn.GRU(hidden_size, hidden_size, n_layers,
                          dropout=(0 if n_layers == 1 else dropout), bidirectional=True)
        elif (self.nn_type == "LSTM"):
          print("You are ready to train or evaluate with an LSTM RNN.")
          self.lstm = nn.LSTM(hidden_size, hidden_size, n_layers, dropout=(0 if n_layers == 1 else dropout), bidirectional=True)

    def forward(self, input_seq, input_lengths, hidden=None):
        # Convert word indexes to embeddings
        print()
        print("WE ARE NOW IN THE FORWARD FUNCTION OF THE ENCODER")
        print()
        embedded = self.embedding(input_seq)
        print("EMBED BEFORE PACKED: ", embedded)
        print("EMBED SIZE BEFORE PACKED: ", embedded.size())
        # Pack padded batch of sequences for RNN module
        packed = nn.utils.rnn.pack_padded_sequence(embedded, input_lengths)
        print("EMBED AFTER PACKED PADDED BATCH: ", packed)
        print("EMBED SIZE AFTER PACKED: ", packed.batch_sizes.size())

        # Forward pass through GRU or uncomment lstm to forward throught lstm
        print()
        print("WE ARE NOW IN THE FORWARD PASS OF THE GRU OR LSTM (STILL IN THE ENCODER)")
        if (self.nn_type == "GRU"):
          print("THIS IS THE OUPTUT OF A PASS THROUGH THE GRU: ")
          outputs, hidden = self.gru(packed, hidden)
          print("GRU OUTPUTS: ", outputs)
          print("GRU OUTPUTS SIZE: ", outputs.batch_sizes.size())
          print("GRU HIDDEN: ", hidden)
          print("GRU HIDDEN SIZE: ", hidden.size())
        elif (self.nn_type == "LSTM"):
          print("THIS IS THE OUPTUT OF A PAS THROUGH THE LSTM: ")
          outputs, hidden = self.lstm(packed, hidden)
          print("LSTM OUTPUTS: ", outputs)
          print("LSTM OUTPUTS SIZE: ", len(outputs))
          print("LSTM HIDDEN TUPLE?!!!!!: ", hidden)
          #print("LSTM HIDDEN SIZE: ", hidden.batch_size.size())
        # Unpack padding
        outputs, _ = nn.utils.rnn.pad_packed_sequence(outputs)
        print("UNPACKED OUTPUTS FROM GRU OR LSTM: ", outputs)
        # Sum bidirectional GRU outputs
        outputs = outputs[:, :, :self.hidden_size] + outputs[:, : ,self.hidden_size:]
        # Return output and final hidden state
        return outputs, hidden

<i>**Attention Layer.**</i>

In [ ]:
# NO MODIFICATION MADE
class Attn(nn.Module):
    def __init__(self, method, hidden_size):
        super(Attn, self).__init__()
        self.method = method
        if self.method not in ['dot', 'general', 'concat']:
            raise ValueError(self.method, "is not an appropriate attention method.")
        self.hidden_size = hidden_size
        if self.method == 'general':
            self.attn = nn.Linear(self.hidden_size, hidden_size) # it's just a feedforward neural network? just a linear transformation. maybe change to CNN????
        elif self.method == 'concat':
            self.attn = nn.Linear(self.hidden_size * 2, hidden_size)
            self.v = nn.Parameter(torch.FloatTensor(hidden_size))

    def dot_score(self, hidden, encoder_output):
        return torch.sum(hidden * encoder_output, dim=2)

    def general_score(self, hidden, encoder_output):
        energy = self.attn(encoder_output)
        return torch.sum(hidden * energy, dim=2)

    def concat_score(self, hidden, encoder_output):
        energy = self.attn(torch.cat((hidden.expand(encoder_output.size(0), -1, -1), encoder_output), 2)).tanh()
        return torch.sum(self.v * energy, dim=2)

    def forward(self, hidden, encoder_outputs):
        # Calculate the attention weights (energies) based on the given method
        if self.method == 'general':
            attn_energies = self.general_score(hidden, encoder_outputs)
        elif self.method == 'concat':
            attn_energies = self.concat_score(hidden, encoder_outputs) # hidden = decoder hidden state, encoder_outputs = all encoder hidden states
        elif self.method == 'dot':
            attn_energies = self.dot_score(hidden, encoder_outputs)

        # Transpose max_length and batch_size dimensions
        attn_energies = attn_energies.t()

        # Return the softmax normalized probability scores (with added dimension)
        print("RETURNING THESE SCORES: ", F.softmax(attn_energies, dim=1).unsqueeze(1))
        return F.softmax(attn_energies, dim=1).unsqueeze(1)

**Decoder.**

In [ ]:
# MODIFICATIONS MADE
class LuongAttnDecoderRNN(nn.Module):
    def __init__(self, attn_model, embedding, nn_type, hidden_size, output_size, n_layers=1, dropout=0.1):
        super(LuongAttnDecoderRNN, self).__init__()

        # Keep for reference
        self.attn_model = attn_model
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.n_layers = n_layers
        self.dropout = dropout
        self.nn_type = nn_type

        # Define layers
        self.embedding = embedding
        self.embedding_dropout = nn.Dropout(dropout)
        if (self.nn_type == "GRU"):
          self.gru = nn.GRU(hidden_size, hidden_size, n_layers, dropout=(0 if n_layers == 1 else dropout))
        elif (self.nn_type == "LSTM"):
          self.lstm = nn.LSTM(hidden_size, hidden_size, n_layers, dropout=(0 if n_layers == 1 else dropout))
        self.concat = nn.Linear(hidden_size * 2, hidden_size)
        self.out = nn.Linear(hidden_size, output_size)
        #self.out = nn.Conv2d(hidden_size, output_size, 5*5)

        self.attn = Attn(attn_model, hidden_size)

    def forward(self, input_step, last_hidden, encoder_outputs):
        print()
        print("WE ARE IN THE DECODER CLASS FORWARD METHOD")
        print("DECODER INPUT_STEP, WAS [1, 1, 1, 1, 1,]", input_step)

        # input_step = one single word from the input sequence batch.
        # last_hidden = the final hidden state layer from the encoder RNN.
        # Note: we run this one step (word) at a time
        # Get embedding of current input word
        embedded = self.embedding(input_step)
        print("EMBEDDING OF CURRENT INPUT WORD: ", embedded)
        embedded = self.embedding_dropout(embedded)
        # get output and hidden states depending on which unit is chosen
        if (self.nn_type == "GRU"):
          rnn_output, hidden = self.gru(embedded, last_hidden)
        elif (self.nn_type == "LSTM"):
          rnn_output, hidden = self.lstm(embedded,last_hidden)
        print("WE HAVE JUST COMPUTED THE GRU OUTPUT FOR EMBEDDED AND LAST_HIDDEN STATE FROM THE ENCODER")

        print()

        print("WE ARE ABOUT TO CALCULATE THE ATTENTION WEIGHTS")
        # Calculate attention weights from the current GRU or LSTM output
        print("WE HAVE ENTERED LUONG ATTENTION CLASS")
        attn_weights = self.attn(rnn_output, encoder_outputs)
        print("ATTENTION WEIGHTS SOFTMAX SCORES SHOULD BE THE SAME AS PREVIOUS PRINT: ", attn_weights)
        # Multiply attention weights to encoder outputs to get new "weighted sum" context vector
        context = attn_weights.bmm(encoder_outputs.transpose(0, 1))
        print("THIS IS THE CONTEXT VECTOR THAT HAS NOW HAD ATTENTION APPLIED TO IT", context)
        # Concatenate weighted context vector and GRU output using Luong eq. 5
        rnn_output = rnn_output.squeeze(0)
        context = context.squeeze(1)
        concat_input = torch.cat((rnn_output, context), 1)
        concat_output = torch.tanh(self.concat(concat_input))
        # Predict next word using Luong eq. 6
        output = self.out(concat_output)
        output = F.softmax(output, dim=1)
        # Return output and final hidden state
        return output, hidden

# **Training Routine**

This section was left largely unchanged apart from adding additional argument to the train functions and giving them the functionality to process datasets with the LSTM architecture. We have commented out the next three code blocks for this demonstration.

<i>**Masked Loss Function.**</i>



In [ ]:

def maskNLLLoss(inp, target, mask):
    nTotal = mask.sum()
    crossEntropy = -torch.log(torch.gather(inp, 1, target.view(-1, 1)).squeeze(1))
    loss = crossEntropy.masked_select(mask).mean()
    loss = loss.to(device)
    return loss, nTotal.item()


<i>**Train Function.**</i>

In [ ]:

def train(input_variable, lengths, target_variable, mask, max_target_len, encoder, decoder, embedding,
          encoder_optimizer, decoder_optimizer, batch_size, clip, MAX_LENGTH):# input_variable is the input tensor (the mini batch)
    print()
    print("WE HAVE ENETERED train()")
    print()
    # Zero gradients
    encoder_optimizer.zero_grad()
    decoder_optimizer.zero_grad()
    print("GRADIENTS HAVE BEEN ZEROED FOR ENCODER AND DECODER")

    # Set device options
    input_variable = input_variable.to(device)
    target_variable = target_variable.to(device)
    #print("INPUT AND TARGET TENSORS ARE NOW ON THE SPECIFIED DEVICE, SUCH AS A GPU")
    mask = mask.to(device)
    # Lengths for rnn packing should always be on the cpu
    lengths = lengths.to("cpu")

    # Initialize variables
    loss = 0
    print_losses = []
    n_totals = 0

    # Forward pass through encoder
    print()
    print("WE ARE ABOUT TO DO A FORWARD PASS THROUGHT THE ENCODER")
    print()
    encoder_outputs, encoder_hidden = encoder(input_variable, lengths)

    print("ENCODER OUTPUTS:", encoder_outputs)
    print("ENCODER HIDDEN: ", encoder_hidden)

    # Create initial decoder input (start with SOS tokens for each sentence)
    decoder_input = torch.LongTensor([[SOS_token for _ in range(batch_size)]]) #SOS_token == 1
    print("INITIAL DECODER INPUT: ", decoder_input)
    decoder_input = decoder_input.to(device)

    # Set initial decoder hidden state to the encoder's final hidden state
    if encoder.nn_type == 'GRU': # (COPIED NEW CODE! 1)
      decoder_hidden = encoder_hidden[:decoder.n_layers]
    elif (encoder.nn_type == "LSTM"):
      encoder_lstm_hidden, encoder_lstm_hid = encoder_hidden
      decoder_lstm_hidden = encoder_lstm_hidden[:decoder.n_layers]
      decoder_lstm_hid = encoder_lstm_hid[:decoder.n_layers]
      decoder_hidden = (decoder_lstm_hidden, decoder_lstm_hid)

    # Determine if we are using teacher forcing this iteration
    use_teacher_forcing = True if random.random() < teacher_forcing_ratio else False

    # Forward batch of sequences through decoder one time step at a time
    timeStep = 1
    print()
    print("WE ARE ABOUT TO ENTER THE DECODER")
    if use_teacher_forcing:
        for t in range(max_target_len):
            print("WE ARE AT TIME STEP: ", timeStep)
            print("WE ARE IN DECODER CLASS FOR THIS TIME STEP WITH TEACHER FORCING")
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden, encoder_outputs
            )
            print("WE HAVE EXITED THE DECODER CLASS FOR THIS TIME STEP")
            print("DECODERS OUTPUT", decoder_output)
            # Teacher forcing: next input is current target
            decoder_input = target_variable[t].view(1, -1)
            # Calculate and accumulate loss
            mask_loss, nTotal = maskNLLLoss(decoder_output, target_variable[t], mask[t])
            loss += mask_loss
            print_losses.append(mask_loss.item() * nTotal)
            n_totals += nTotal
            timeStep = timeStep + 1
    else:
        for t in range(max_target_len):
            print("WE ARE AT TIME STEP: ", timeStep)
            print("WE ARE IN DECODER CLASS FOR THIS TIME STEP WITHOUT TEACHER FORCING")
            decoder_output, decoder_hidden = decoder(
                decoder_input, decoder_hidden, encoder_outputs
            )
            print("WE HAVE EXITED THE DECODER CLASS FOR THIS TIME STEP")
            print("DECODERS OUTPUT", decoder_output)
            # No teacher forcing: next input is decoder's own current output
            _, topi = decoder_output.topk(1)
            decoder_input = torch.LongTensor([[topi[i][0] for i in range(batch_size)]])
            decoder_input = decoder_input.to(device)
            # Calculate and accumulate loss
            mask_loss, nTotal = maskNLLLoss(decoder_output, target_variable[t], mask[t])
            loss += mask_loss
            print_losses.append(mask_loss.item() * nTotal)
            n_totals += nTotal
            timeStep = timeStep + 1

    # Perform backpropatation
    loss.backward()

    # Clip gradients: gradients are modified in place
    _ = nn.utils.clip_grad_norm_(encoder.parameters(), clip)
    _ = nn.utils.clip_grad_norm_(decoder.parameters(), clip)

    # Adjust model weights
    encoder_optimizer.step()
    decoder_optimizer.step()
    print("WE HAVE PERFORMED BACK PROP AND ADUSTED THE WEIGHTS WITH .STEP()")
    return sum(print_losses) / n_totals


<i>**Training Iterations.**</i>

Commented out as training is completed.

In [ ]:

def trainIters(model_name, voc, pairs, encoder, decoder, encoder_optimizer, decoder_optimizer, embedding, encoder_n_layers, decoder_n_layers, save_dir, n_iteration, batch_size,
               print_every, save_every, clip, corpus_name, loadFilename, sent_len, drop_o, attention, network):
    print()
    print("WE HAVE ENTERED trainIters")
    print()
    # Load batches for each iteration
    training_batches = [batch2TrainData(voc, [random.choice(pairs) for _ in range(batch_size)])
                      for _ in range(n_iteration)]



    # Initializations
    print('Initializing ...')
    start_iteration = 1
    print_loss = 0
    if loadFilename:
        start_iteration = checkpoint['iteration'] + 1

    # Training loop
    print("Training...")
    for iteration in range(start_iteration, n_iteration + 1):
        training_batch = training_batches[iteration - 1] #  get a training batch of size batch_size
        # Extract fields from batch
        input_variable, lengths, target_variable, mask, max_target_len = training_batch
        print("ITERATION NUMBER ", iteration)
        print("INPUT TENSOR: ", input_variable)
        print("LENGTH OF INPUT TENSOR SENTENCES", lengths)
        print()
        print("TARGET TENSOR:", target_variable)
        print("MASK: ", mask)
        print("MAX TARGET LENGTH", max_target_len)

        # Run a training iteration with batch
        loss = train(input_variable, lengths, target_variable, mask, max_target_len, encoder,
                     decoder, embedding, encoder_optimizer, decoder_optimizer, batch_size, clip, sent_len)
        print_loss += loss

        # Print progress
        if iteration % print_every == 0:
            print_loss_avg = print_loss / print_every
            print("Iteration: {}; Percent complete: {:.1f}%; Average loss: {:.4f}".format(iteration, iteration / n_iteration * 100, print_loss_avg))
            print_loss = 0

        # Save checkpoint
        if (iteration % save_every == 0):
            directory = os.path.join(save_dir, model_name, corpus_name, '{}-{}_{}_{}_{}_{}_{}_{}'.format(encoder_n_layers, decoder_n_layers, hidden_size, learning_rate, sent_len, drop_o, attention, network)) #sent_len, drop_out
            if not os.path.exists(directory):
                os.makedirs(directory)
            torch.save({
                'iteration': iteration,
                'en': encoder.state_dict(),
                'de': decoder.state_dict(),
                'en_opt': encoder_optimizer.state_dict(),
                'de_opt': decoder_optimizer.state_dict(),
                'loss': loss,
                'voc_dict': voc.__dict__,
                'embedding': embedding.state_dict()
            }, os.path.join(directory, '{}_{}.tar'.format(iteration, 'checkpoint')))


## **Model Run.**

Here we get the model states from the four pre-trained versions of the DocBot and load them in for evaluation.

In [ ]:
# Configure models
model_name = 'cb_model'
attn_model = 'dot'
#attn_model = 'general'
#attn_model = 'concat'
#hidden_size_500 =  500 #def=500
hidden_size_200 = 4
#hidden_size1 = 200
encoder_n_layers = 1 #def=2
decoder_n_layers = 1 #def=2
#encoder_n_layers1 = 1 #def=2
#decoder_n_layers1 = 1 #def=2
dropout = 0.1
batch_size = 5 # def = 64
network_type1 = "GRU"
network_type2 = "LSTM"
l_rate = 0.0001
sent_len1 = 100
sent_len2 = 70

checkpoint_iter_120k = 120000
checkpoint_iter_100k = 100000
checkpoint_iter_50k = 50000
checkpoint_iter_80k = 80000


# create vocabularies

save_dir = os.path.join(corpus, "Model Save Folder")
voc, pairs = loadPrepareData(corpus, corpus_name, training_dataset, save_dir, 10)

print("PAIRS: ")
for line in pairs[:4]:
  print(line)
#voc2, pairs2 = loadPrepareData(corpus, corpus_name, training_dataset, save_dir, 70) # this is for one lstm model

## Only necessary for training.
swap(pairs)
 #swap(pairs2)

#pairs1 = trimRareWords(voc1, pairs1, MIN_COUNT)
#pairs2 = trimRareWords(voc2, pairs2, MIN_COUNT)

#batches1 = batch2TrainData(voc1, [random.choice(pairs1) for _ in range(small_batch_size)])
#batches2 = batch2TrainData(voc2, [random.choice(pairs2) for _ in range(small_batch_size)])

## Set checkpoint to load from; set to None if starting from scratch
loadFilename = None

# load trained models for evaluating or resumption of training

# GRU, 120,000 iterations completed, 2 layers, 500 hidden size
#load_trained_model1 = os.path.join(save_dir, model_name, corpus_name,
                   #        '{}-{}_{}_{}_{}_{}_{}_{}'.format(encoder_n_layers2, decoder_n_layers2, hidden_size_500, l_rate, sent_len1, dropout, attn_model, network_type1),
                    #        '{}_checkpoint.tar'.format(checkpoint_iter_120k))

# LSTM, 50,000 iterations completed, 2 layers, 500 hidden size 70 words
#load_trained_model2 = os.path.join(save_dir, model_name, corpus_name,
                      #     '{}-{}_{}_{}_{}_{}_{}_{}'.format(encoder_n_layers2, decoder_n_layers2, hidden_size_500, l_rate, sent_len2, dropout, attn_model, network_type2),
                      #      '{}_checkpoint.tar'.format(checkpoint_iter_50k))


# LSTM, 100,000 iterations completed, 2 layers, 500 hidden size 100 words
#load_trained_model3 = os.path.join(save_dir, model_name, corpus_name,
    #                       '{}-{}_{}_{}_{}_{}_{}_{}'.format(encoder_n_layers2, decoder_n_layers2, hidden_size_500, l_rate, sent_len1, dropout, attn_model, network_type2),
     #                       '{}_checkpoint.tar'.format(checkpoint_iter_100k))

# GRU, 100,000 iterations completed, 1 layers, 200 hidden size 100 words
#load_trained_model4 = os.path.join(save_dir, model_name, corpus_name,
        #                   '{}-{}_{}_{}_{}_{}_{}_{}'.format(encoder_n_layers1, decoder_n_layers1, hidden_size_200, l_rate, sent_len1, dropout, attn_model, network_type1),
         #                   '{}_checkpoint.tar'.format(checkpoint_iter_100k))



# Load model 1
#if load_trained_model1:
    # If loading on same machine the model was trained on
    #checkpoint1 = torch.load(load_trained_model1)
    # If loading a model trained on GPU to CPU
 #   checkpoint1 = torch.load(load_trained_model1, map_location=torch.device('cpu'))
  #  encoder_sd1 = checkpoint1['en']
 #   decoder_sd1 = checkpoint1['de']
  #  encoder_optimizer_sd1 = checkpoint1['en_opt']
  #  decoder_optimizer_sd1 = checkpoint1['de_opt']
  #  embedding_sd1 = checkpoint1['embedding']
  #  voc1.__dict__ = checkpoint1['voc_dict']

# Load model 2
#if load_trained_model2:
    # If loading on same machine the model was trained on
    #checkpoint2 = torch.load(load_trained_model2)
    # If loading a model trained on GPU to CPU
   # checkpoint2 = torch.load(load_trained_model2, map_location=torch.device('cpu'))
  #  encoder_sd2 = checkpoint2['en']
  #  decoder_sd2 = checkpoint2['de']
   # encoder_optimizer_sd2 = checkpoint2['en_opt']
   # decoder_optimizer_sd2 = checkpoint2['de_opt']
   # embedding_sd2 = checkpoint2['embedding']
  #  voc2.__dict__ = checkpoint2['voc_dict']

# Load model 3
#if load_trained_model3:
    # If loading on same machine the model was trained on
    #checkpoint3 = torch.load(load_trained_model3)
    # If loading a model trained on GPU to CPU
   # checkpoint3 = torch.load(load_trained_model3, map_location=torch.device('cpu'))
   # encoder_sd3 = checkpoint3['en']
   # decoder_sd3 = checkpoint3['de']
   # encoder_optimizer_sd3 = checkpoint3['en_opt']
   # decoder_optimizer_sd3 = checkpoint3['de_opt']
   # embedding_sd3 = checkpoint3['embedding']
   # voc1.__dict__ = checkpoint3['voc_dict']

# Load model 4
#if load_trained_model4:
    # If loading on same machine the model was trained on
    #checkpoint4 = torch.load(load_trained_model4)
    # If loading a model trained on GPU to CPU
   # checkpoint4 = torch.load(load_trained_model4, map_location=torch.device('cpu'))
   # encoder_sd4 = checkpoint4['en']
   # decoder_sd4 = checkpoint4['de']
   # encoder_optimizer_sd4 = checkpoint4['en_opt']
   # decoder_optimizer_sd4 = checkpoint4['de_opt']
   # embedding_sd4 = checkpoint4['embedding']
   # voc1.__dict__ = checkpoint4['voc_dict']

if loadFilename:
    # If loading on same machine the model was trained on
    checkpoint = torch.load(loadFilename)
    # If loading a model trained on GPU to CPU
    #checkpoint = torch.load(loadFilename, map_location=torch.device('cpu'))
    encoder_sd = checkpoint['en']
    decoder_sd = checkpoint['de']
    encoder_optimizer_sd = checkpoint['en_opt']
    decoder_optimizer_sd = checkpoint['de_opt']
    embedding_sd = checkpoint['embedding']
    voc.__dict__ = checkpoint['voc_dict']

print('Building encoders and decoders ...')
# Initialize word embeddings
embedding = nn.Embedding(voc.num_words, hidden_size_200) # this is where all embeddings are learned
if loadFilename:
    embedding.load_state_dict(embedding_sd)
# Initialize encoder & decoder models
encoder = EncoderRNN(hidden_size_200, embedding, network_type1, encoder_n_layers, dropout) #network_type
decoder = LuongAttnDecoderRNN(attn_model, embedding, network_type1, hidden_size_200, voc.num_words, decoder_n_layers, dropout) #network_type
if loadFilename:
    encoder.load_state_dict(encoder_sd)
    decoder.load_state_dict(decoder_sd)
# Use appropriate device
encoder = encoder.to(device)
decoder = decoder.to(device)

Start preparing training data ...
Reading lines...
Read 74368 sentence pairs
Trimmed to 3214 sentence pairs
Counting words...
Counted words: 1646
PAIRS: 
['oh uh no pain with abduction .', 'does that hurt ?']
['um yeah well say that there is swelling .', 'ok how about smell ?']
['no .', 'or any any weight loss ?']
['its been dry .', 'any difficulties breathing ?']
Building encoders and decoders ...
You are ready to train or evaluate with a GRU RNN.


<i>**Set training perameters.**</i>

Commented our for demonstration and evaluation.

In [ ]:

# Configure training/optimization
clip = 50.0
teacher_forcing_ratio = 1.0
learning_rate = 0.0001
decoder_learning_ratio = 5.0
n_iteration = 2
print_every = 1
save_every = 500
sent_len = 10
nn_type = "GRU" # GRU or LSTM

# Ensure dropout layers are in train mode
encoder.train()
decoder.train()

# Initialize optimizers
print('Building optimizers ...')
encoder_optimizer = optim.Adam(encoder.parameters(), lr=learning_rate)
decoder_optimizer = optim.Adam(decoder.parameters(), lr=learning_rate * decoder_learning_ratio)
if loadFilename:
    encoder_optimizer.load_state_dict(encoder_optimizer_sd)
    decoder_optimizer.load_state_dict(decoder_optimizer_sd)

# If you have cuda, configure cuda to call
for state in encoder_optimizer.state.values():
    for k, v in state.items():
        if isinstance(v, torch.Tensor):
            state[k] = v.cuda()

for state in decoder_optimizer.state.values():
    for k, v in state.items():
        if isinstance(v, torch.Tensor):
            state[k] = v.cuda()

# Run training iterations

print("Starting Training!")
trainIters(model_name, voc, pairs, encoder, decoder, encoder_optimizer, decoder_optimizer,
           embedding, encoder_n_layers, decoder_n_layers, save_dir, n_iteration, batch_size,
           print_every, save_every, clip, corpus_name, loadFilename, sent_len, dropout, attn_model, nn_type)


Building optimizers ...
Starting Training!

WE HAVE ENTERED trainIters

Initializing ...
Training...
ITERATION NUMBER  1
INPUT TENSOR:  tensor([[ 21, 254, 513,  72, 127],
        [127, 696, 250, 223,  72],
        [ 72, 408,  20, 121,  45],
        [346, 304, 514, 154,  26],
        [341,  28, 404, 280, 330],
        [ 23, 219, 515, 281,  13],
        [ 37, 169, 516,   9,   2],
        [776, 310, 517,   2,   0],
        [ 13, 311,   2,   0,   0],
        [  2,   2,   0,   0,   0]])
LENGTH OF INPUT TENSOR SENTENCES tensor([10, 10,  9,  8,  7])

TARGET TENSOR: tensor([[  4, 694, 512,   5,   5],
        [  5, 695,   2,  64,   9],
        [561,   2,   0,  95,   2],
        [  9,   0,   0,  55,   0],
        [  2,   0,   0, 279,   0],
        [  0,   0,   0,   9,   0],
        [  0,   0,   0,  64,   0],
        [  0,   0,   0, 131,   0],
        [  0,   0,   0,   9,   0],
        [  0,   0,   0,   2,   0]])
MASK:  tensor([[ True,  True,  True,  True,  True],
        [ True,  True,  True,  T

# **Evaluation.**

We will conduct two types of evaluation methods, a quantitative evaluation method using the Bilingual Evaluation Understudy (Bleu), two different cosine similarity tests and a readability test. We will comment on the different model’s overall loss achieved, how many iterations were used to train each model, the size of the hidden states and how many layers were used for the encoder and decoder. For the quantitative evaluation we will use the test_set that was partitioned off earlier in the program.

As noted in the executive summary, a lot of models were trained with various combinations of different hyperparameter values. Of which, we demonstrate four of the best performing models in terms of the legible responses the DocBot produced.
Models that were trained with a low number of iterations (<50,000) and with small sentence consideration (<50 tokens) did not produce satisfactory results. The datasets used contained very long sentence lengths due to the nature of the conversations (medical queries). If we had demonstrated a well-trained model with very low loss thanks to a small sentence length (as seen in the default version of this model with the movie dialog), the responses were not long enough to give any useful information whatsoever. Furthermore, given that the majority of sentences in the training set were >50 words in length, using small sentence lengths substantially reduced the number of lines to train the model and resulted in small vocabularies. After trial and error, we set a minimum sentence length of 70. One of the four models demonstrated in this notebook used sentence length of 70, the other three used 100 words. We had to balance the sentence length with the time it took to train the model.

The second evaluation method used is qualitative in nature. This is the part where will talk to the DocBot manually. For this evaluation we created a made-up symptom dataset informed by the NHS website. We also evaluate the model by asking it medical queries the author devised. This part will unstructured, we are just investigating which model produced more readable and relevant responses.

Before beginning our evaluation we will implement some helper evaluation functions, in addition to the one that was provided with the base code.



<i>**Greedy Search**</i>

This class simply extracts the next word in the output sequence with the highest probability. No changes were made apart from having it work with the LSTM RNN.

In [ ]:
class GreedySearchDecoder(nn.Module):
    def __init__(self, encoder, decoder):
        super(GreedySearchDecoder, self).__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, input_seq, input_length, max_length):
        # Forward input through encoder model


        # Prepare encoder's final hidden layer to be first hidden input to the decoder
        encoder_outputs, encoder_hidden = self.encoder(input_seq, input_length)
        if (self.encoder.nn_type == "GRU"):
          decoder_hidden = encoder_hidden[:self.decoder.n_layers]
        elif (self.encoder.nn_type == "LSTM"):
          encoder_lstm_hidden, encoder_lstm_hid= encoder_hidden
          decoder_lstm_hidden = encoder_lstm_hidden[:self.decoder.n_layers]
          decoder_lstm_hid = encoder_lstm_hid[:self.decoder.n_layers]
          decoder_hidden = (decoder_lstm_hidden, decoder_lstm_hid)

        # Initialize decoder input with SOS_token
        decoder_input = torch.ones(1, 1, device=device, dtype=torch.long) * SOS_token
        # Initialize tensors to append decoded words to
        all_tokens = torch.zeros([0], device=device, dtype=torch.long)
        all_scores = torch.zeros([0], device=device)
        # Iteratively decode one word token at a time
        for _ in range(max_length):
            # Forward pass through decoder
            #print(decoder_input.size())
            #print(decoder_hidden.size())
            #print("decoder_input type:", type(decoder_input))
            #print("decoder_hidden type:", type(decoder_hidden))
            decoder_output, decoder_hidden = self.decoder(decoder_input, decoder_hidden, encoder_outputs)
            # Obtain most likely word token and its softmax score
            decoder_scores, decoder_input = torch.max(decoder_output, dim=1) # torch.max returns the max value of decoder_output which is a tensor softmax probability distribution, we simply pick the word with the max probability, i.e, the one closes to 1.
            # Record token and score
            all_tokens = torch.cat((all_tokens, decoder_input), dim=0)
            all_scores = torch.cat((all_scores, decoder_scores), dim=0)
            # Prepare current token to be next decoder input (add a dimension)
            decoder_input = torch.unsqueeze(decoder_input, 0)
        # Return collections of word tokens and scores
        return all_tokens, all_scores



<i>**Evaluate Function.**</i>

We had to make some significant changes to the original evaluate function.

We have created another evaluate function for the quantitative evaluation phase of this project (see section "Batch Evaluation Function"). The evaluate function below is only used for manual evaluation, i.e, talking to the chatbot by typing at a prompt.

The sentences processed by this model are very long, as such we simply delete any words not in our vocabulary in order to achieve two goals. 1) the user can get a response regardless of what they type. We acknowledge of course there is an inherent risk that important words that carry significant meaning in the input sentence may be lost. We feel that given this modest piece of work we can at this stage take that risk because the sentences are so long that there should still be enough information (words) in the sentence for the model to extract some semantical information. We have implemented a method within the evaluateInput() function to delete words that are not in the vocabulary. As noted earlier, we use the get() method from the vocabulary class to achieve this and override the error message that was initially returned when an unknown word was encountered.

We have also implemented the functionality to delete repeated phrases that the DocBot outputted using the regular expression module, this functionality on lines 50 to 53 searches the output sentence for any phrases consecutively repeated more than once and then deletes all but the first occurrence of it. Finally, we use the textwrap library to keep the very long output responses within the output window so that the user doesn't have to scroll to the right to read the entire response.


In [ ]:
def evaluate(encoder, decoder, searcher, voc, sentence, MAX_LENGTH):

    ### Format input sentence as a batch
    # words -> indexes
    indexes_batch = [indexesFromSentence(voc, sentence)]
    # Create lengths tensor
    lengths = torch.tensor([len(indexes) for indexes in indexes_batch])
    # Transpose dimensions of batch to match models' expectations
    input_batch = torch.LongTensor(indexes_batch).transpose(0, 1)
    # Use appropriate device
    input_batch = input_batch.to(device)
    lengths = lengths.to("cpu")
    # Decode sentence with searcher
    tokens, scores = searcher(input_batch, lengths, MAX_LENGTH)
    # indexes -> words
    decoded_words = [voc.index2word[token.item()] for token in tokens] # if word[i].count > 4, delete it!
    return decoded_words


def evaluateInput(encoder, decoder, searcher, voc, MAX_LENGTH):
    print("This evaluation is with the", encoder.nn_type + " " + "RNN")
    testbool = True
    input_sentence = ''
    while(1):
        try:
            # Get input sentence
            input_sentence = input('> ')
            # Check if it is quit case
            if input_sentence == 'q' or input_sentence == 'quit': break
            # Normalize sentence
            input_sentence = normalizeString(input_sentence)
            # check if any words are not in vocabularly. If so, remove them.
            for index in range(len(input_sentence)):
              tokenize_sent = word_tokenize(input_sentence)
              for j in range(len(tokenize_sent)):
                wordInVoc = voc.get(tokenize_sent[j])
                if (wordInVoc == None):
                  #delete word from input_sentence
                  input_sentence = input_sentence.replace(tokenize_sent[j], "")
            # Evaluate sentence
            output_words = evaluate(encoder, decoder, searcher, voc, input_sentence, MAX_LENGTH)
            # Format and print response sentence
            output_words[:] = [x for x in output_words if not (x == 'EOS' or x == 'PAD')]
            # remove unwanted repetition of phrases
            try:
              output_words_toString = ' '.join(output_words)
              re_remove = re.search(r'\b(\w+(?:\s*\w*))\s+\1\b', output_words_toString).groups()[0]
              out = re.sub(f'({re_remove}\s*)+', re_remove + " ", output_words_toString)
              new_output_words = out.split()
              #print("\033[1m" + "DocBot:" + "\033[0m", ' '.join(new_output_words))
              output_text = "\033[1m" + "DocBot:" + "\033[0m" + ' '.join(new_output_words)
              wrapped_text = textwrap.wrap(output_text, width=125)
              print('\n'.join(wrapped_text))
              print()
            # if there are no repeated phrases, simply print the original output_words
            except AttributeError:
              # make sure output fits the output window
              output_text = "\033[1m" + "DocBot:" + "\033[0m" + ' '.join(output_words)
              wrapped_text = textwrap.wrap(output_text, width=125)
              print('\n'.join(wrapped_text))
              print()
             #print("\033[1m" + "DocBot:" + "\033[0m", ' '.join(output_words))
        # now there should be no need to enter this except statement. We leave it here just in case.
        except KeyError:
            testBool = False



<i>**Initialize Evaluation Mode for the Models chosen for Demonstration**</i>.

In [ ]:
# Set dropout layers to eval mode for all trained models
# GRU model trained with 100 words and 120,000 iterations
encoder1.eval()
decoder1.eval()

# LSTM model trained with 70 words and 50,000 iterations
encoder2.eval()
decoder2.eval()

# LSTM model trained with 100 words and 100,000 iterations
encoder3.eval()
decoder3.eval()

# GRU model trained with 100 words, 100,000 iterations, 1 hidden layer and a 200 hidden_size
encoder4.eval()
decoder4.eval()

# Initialize search module
searcher1 = GreedySearchDecoder(encoder1, decoder1)
searcher2 = GreedySearchDecoder(encoder2, decoder2)
searcher3 = GreedySearchDecoder(encoder3, decoder3)
searcher4 = GreedySearchDecoder(encoder4, decoder4)

## **Quantitative Evaluation**

Although the default model did not evaluate with a test dataset, we have created one for two reasons. Firstly, for an extensive quantitative evaluation we do not want to manually type sentences we made up ourselves (this is more appropriate for a qualitative analysis). Secondly, this is a domain specific attempt at creating a chatbot, a domain in which we have limited knowledge of. By partitioning off a testing set comprised of 10% of the final dataset created earlier, we can solve both of these problems.

Another advantage to creating a testing set is that we can be sure the model hasn't seen these input questions. Even though the benefit to the evaluation of the model is not quite the same as a classification task.

For the quantitative part of the evaluation, we want to input multiple query sentences from the test_set simultaneously in order obtain BLEU and cosine similarity scores. First, we will define some helper functions to extract batches of test pairs from the test_set that will be fed into an evaluation function which will return all generated responses by the four models in lists so that we can compare them with the target responses in the test_set.


<i>**Extract test sentence pairs then split test pairs into two lists, inputs and targets.**</i>

We extract two different lists of inputs and targets, a small batch with 20 lines and a large batch with 100 lines. The large_inputs and large_targets will be used for the readablitity part of the quantitative test.

In [ ]:
test_set_file = os.path.join(corpus, "test_set.txt")
def extractTestPairs(test_set, num_of_pairs):
  test_p = []
  p = open(test_set).\
    read().strip().split('\n')
  test_p = [[normalizeString(s) for s in l.split('\t')] for l in p[:num_of_pairs]] # [-x:] = start from end, take x lines
  return test_p

test_pairs = extractTestPairs(test_set_file, 20)
large_test_pairs = extractTestPairs(test_set_file, 100)

def splitTestPairs(testPairs):
  inputs = []
  targets = []
  for i in range(len(testPairs)):
    inputs.append(testPairs[i][1])
    targets.append(testPairs[i][0])
  return inputs, targets

inputs, targets = splitTestPairs(test_pairs)
large_inputs, large_targets = splitTestPairs(large_test_pairs)

<i>**Print the inputs and targets.**</i>

In [ ]:
for i in range(10):
  print("INPUT", i+1, large_inputs[i])
  print("TARGET", i+1, large_targets[i])

INPUT 1 hi i ate something an i developed rash on my body . nothing serious just really small red dots and bit itchy . i don t have any other food allergy or food poisoning symptoms . . .it happened before after i ate meat medium beef for ex . what can i use to apply on my skin ? thank you
TARGET 1 hello . thank you for using ask a doctor service . i have gone through your query and here is my advice .you seem to have hives . hives present as itchy welts and redness flare . allergic reaction to a food or food preservatives is a common cause of hives . other causes include infection drugs alcohol etc . i suggest you to take an oral antihistamine e .g tab cetrizine mg once a day for a few days . in addition i suggest you to apply a soothing lotion e .g calamine lotion to help reduce itching and flare .hope i have answered your querylet me know if you need any further assistance .
INPUT 2 i have these small tiny bumps only elbows . had one that was a bump for a long time then turned into 

<i>**Delete words from the targets list that are not in the vocabularly**</i>

In [ ]:
#delete words from targets not in vocabularly
target_sentences = []
for index in range(len(targets)):
        target_sent = targets[index]
        tokenize_sent = word_tokenize(target_sent)
        for j in range(len(tokenize_sent)):
          wordInVoc = voc1.get(tokenize_sent[j])
          if (wordInVoc == None):
            #delete word from input_sent
            target_sent = target_sent.replace(tokenize_sent[j], "")
        target_sentences.append(target_sent)

<i>**Batch Evaluate Function.**</i>

We created an evaluate function to obtain the generated responses from the trained chatbot. These responses, along with the targets list, will be used to conduct a quantitative evaluation as described earlier.

This evaluate function is different from evaluateInput() function. That function is for manually entering queries for the DocBot to answer. This evaluateTestBatch() function allows us to evaluate and receive responses to multiple input queries which we can then run quantitative tests on. It returns a list of responses in which the indexes of the responses correspond with the indexes of the target sentences from the test_data set, thus we can compare them.

The evaluateTestBatch() function again takes care of the problem of what the DocBot does when encountering aunknown words. By default, the chatbot simply printed a message to the user prompting them to input another query. The trouble with this in terms of performing extensive testing is that there may be a lot of words that the chatbot has never seen before, in both testing datasets and from user input. We found it more satisfactory both from a user point of view and in order to test the DocBot to simply delete unknown words from input sentences and see if the DocBot can still return a coherent and/or relevant sentence.

As before, we also implement the ability to remove repeated phrases from the DocBot responses.



In [ ]:
def evaluateTestBatch(encoder, decoder, searcher, voc, sentences, max_l):
    print("This test_batch evaluation is with the", encoder.nn_type + " " + "RNN")
    bot_responses = []
    wordInVoc = ""
    for index in range(len(sentences)):
        input_sent = sentences[index]
        tokenize_sent = word_tokenize(input_sent)
        for j in range(len(tokenize_sent)):
          wordInVoc = voc.get(tokenize_sent[j])
          if (wordInVoc == None):
            #delete word from input_sent
            input_sent = input_sent.replace(tokenize_sent[j], "")
        output_words = evaluate(encoder, decoder, searcher, voc, input_sent, max_l)
        output_words[:] = [x for x in output_words if not (x == 'EOS' or x == 'PAD')]
        # remove unwanted repetition of phrases if they exist, otherwise handle the AttributeError that would occur by simply appending the output words
        try:
          output_words_toString = ' '.join(output_words)
          re_remove = re.search(r'\b(\w+(?:\s*\w*))\s+\1\b', output_words_toString).groups()[0]
          out = re.sub(f'({re_remove}\s*)+', re_remove + " ", output_words_toString)
          new_output_words = out.split()
          bot_responses.append(new_output_words)
        except AttributeError:
          bot_responses.append(output_words)
    return bot_responses

We will now evaluate the DocBot by inputting 20 sentences from the test_set (inputs) and saving the generated response lists into the four different models which we are testing.

The four models and their hyperperameters can be found in the table below:

| Model name | Number of iterations | Loss | Hidden size | Encoder layers | Decoder layers | Sentence length | Neural network type |
| --- | --- | --- | --- | --- | --- | --- | --- |
| gru_model_120k | 120000 | 0.4708 | 500 | 2 | 2 | 100 | GRU |
| gru_model_100k | 100000 | 1.8512 | 200 | 1 | 1 | 100 | GRU |
| lstm_model_50k | 50000 | 0.0932 | 500 | 2 | 2 | 70 | LSTM |
| lstm_model_100k | 100000 | 0.2145 | 500 | 2 | 2 | 100 | LSTM |

In [ ]:
gru_model_120k = evaluateTestBatch(encoder1, decoder1, searcher1, voc1, inputs, 100)
gru_model_100k = evaluateTestBatch(encoder4, decoder4, searcher4, voc1, inputs, 100)
lstm_model_50k = evaluateTestBatch(encoder2, decoder2, searcher2, voc2, inputs, 70) # second vocab used for 70 words length
lstm_model_100k = evaluateTestBatch(encoder3, decoder3, searcher3, voc1, inputs, 100)


This test_batch evaluation is with the GRU RNN
This test_batch evaluation is with the GRU RNN
This test_batch evaluation is with the LSTM RNN
This test_batch evaluation is with the LSTM RNN


**Print one of the generated responses.**

As can be seen in the output below, we retain the tokenized output as we are only using these batches of generated responses to perform quantative tests on, all of which require tokenized input.

In [ ]:
for i in range(20):
  print("Input", i+1, inputs[i])
  print("gru120k response", i+1, lstm_model_100k[i])
  print("NEXT PAIR")


Input 1 hi i ate something an i developed rash on my body . nothing serious just really small red dots and bit itchy . i don t have any other food allergy or food poisoning symptoms . . .it happened before after i ate meat medium beef for ex . what can i use to apply on my skin ? thank you
gru120k response 1 ['hello', 'and', 'thanks', 'for', 'your', 'question', '.', 'it', 'is', 'a', 'major', 'problem', '.', 'it', 'could', 'be', 'a', 'skin', 'reaction', '.', 'as', 'you', 'describe', 'the', 'rash', 'a', 'lot', 'will', 'have', 'a', 'lot', 'worse', 'and', 'will', 'take', 'a', 'few', 'more', 'place', '.', 'also', 'you', 'can', 'continue', 'the', 'medications', 'as', 'per', 'the', 'doctor', 'for', 'you', '.', 'hope', 'i', 'have', 'answered', 'your', 'query', '.', 'let', 'me', 'know', 'if', 'i', 'can', 'assist', 'you', 'further', '.', 'good', 'health', '.', 'good', 'health', '.', '.', '.', 'good', 'health', '.', 'sometimes', 'take', 'a', 'body', 'life', 'at', 'bed', 'care', '.', 'regards', 'd

**BLEU Score.**

Although the BLEU evaluation tool was created to aid in machine translation evaluation, it can be applied to an array of NLP applications to check for sentence similiarity<sup>[7][8]</sup>.

We do not expect good results from this test given the size of the sentence responses and the generative conversational model that has been used in this study. Even two doctors describing and diagnosing the same condition may produce sentences that look very differen in terms of the words used. In short, a poor BLEU score does not necessarily mean that the DocBot responses are not fluent or relevant. A qualitiative evalution is more appropriate to answer questions about fluency and relevance. We consider anything that is not a total mismatch (a score of 0.0) to be of interest for our purposes.

As noted previously, all of the quantitative tests performed in this section require tokenized input. The target_sentences are not tokenized so we will do that first within the BLEU function below.

Due to the number of sentences considered and the number of words in each sentence, we take the average BLEU score for each n-gram processed, here we calculate n=1 upto n=4, which is standard when calculating BLEU scores. n > 4 would not produce any useful information as we are not trying to output generated responses that are word for words matches to the target_sentences from the test_set.



In [ ]:
tokenized_targets = []
for line in target_sentences:
  tokenized_targets.append(word_tokenize(line))

def getBleuScores(bot_response_list):
  smooth = SmoothingFunction().method4
  score_n1 = score_n2 = score_n3 = score_n4 = 0.0
  generated_responses_length = len(bot_response_list)
  for index in range(generated_responses_length):
    score_n1 += sentence_bleu(tokenized_targets[index], bot_response_list[index], weights=(1, 0, 0, 0))
    score_n2 += sentence_bleu(tokenized_targets[index], bot_response_list[index], weights=(0.5, 0.5, 0, 0), smoothing_function=smooth)
    score_n3 += sentence_bleu(tokenized_targets[index], bot_response_list[index], weights=(0.33, 0.33, 0.33, 0), smoothing_function=smooth)
    score_n4 += sentence_bleu(tokenized_targets[index], bot_response_list[index], weights=(0.25, 0.25, 0.25, 0.25), smoothing_function=smooth)

  return score_n1, score_n2, score_n3, score_n4

generated_responses_length = len(gru_model_120k)
score1, score2, score3, score4 = getBleuScores(gru_model_120k)
score5, score6, score7, score8 = getBleuScores(lstm_model_50k)
score9, score10, score11, score12 = getBleuScores(lstm_model_100k)
score13, score14, score15, score16 = getBleuScores(gru_model_100k)

print("")
print("1-gram cumulative score average for model 1 responses:", "{:.4f}".format(score1/generated_responses_length))
print("2-gram cumulative score average for model 1 responses:", "{:.4f}".format(score2/generated_responses_length))
print("3-gram cumulative score average for model 1 responses:", "{:.4f}".format(score3/generated_responses_length))
print("4-gram cumulative score average for model 1 responses:", "{:.4f}".format(score4/generated_responses_length))
print("")
print("1-gram cumulative score average for model 2 responses:", "{:.4f}".format(score5/generated_responses_length))
print("2-gram cumulative score average for model 2 responses:", "{:.4f}".format(score6/generated_responses_length))
print("3-gram cumulative score average for model 2 responses:", "{:.4f}".format(score7/generated_responses_length))
print("4-gram cumulative score average for model 2 responses:", "{:.4f}".format(score8/generated_responses_length))
print("")
print("1-gram cumulative score average for model 3 responses:", "{:.4f}".format(score9/generated_responses_length))
print("2-gram cumulative score average for model 3 responses:", "{:.4f}".format(score10/generated_responses_length))
print("3-gram cumulative score average for model 3 responses:", "{:.4f}".format(score11/generated_responses_length))
print("4-gram cumulative score average for model 3 responses:", "{:.4f}".format(score12/generated_responses_length))
print("")
print("1-gram cumulative score average for model 4 responses:", "{:.4f}".format(score13/generated_responses_length))
print("2-gram cumulative score average for model 4 responses:", "{:.4f}".format(score14/generated_responses_length))
print("3-gram cumulative score average for model 4 responses:", "{:.4f}".format(score15/generated_responses_length))
print("4-gram cumulative score average for model 4 responses:", "{:.4f}".format(score16/generated_responses_length))


1-gram cumulative score average for model 1 responses: 0.0451
2-gram cumulative score average for model 1 responses: 0.0164
3-gram cumulative score average for model 1 responses: 0.0098
4-gram cumulative score average for model 1 responses: 0.0060

1-gram cumulative score average for model 2 responses: 0.0561
2-gram cumulative score average for model 2 responses: 0.0217
3-gram cumulative score average for model 2 responses: 0.0131
4-gram cumulative score average for model 2 responses: 0.0080

1-gram cumulative score average for model 3 responses: 0.0480
2-gram cumulative score average for model 3 responses: 0.0175
3-gram cumulative score average for model 3 responses: 0.0104
4-gram cumulative score average for model 3 responses: 0.0064

1-gram cumulative score average for model 4 responses: 0.0442
2-gram cumulative score average for model 4 responses: 0.0158
3-gram cumulative score average for model 4 responses: 0.0094
4-gram cumulative score average for model 4 responses: 0.0057


From the results above we see we are getting scroes very close to zero, as we expected.

<i>**Remove Stopwords.**</i>

Before moving onto the cosine similarity tests, we remove common words that lend no meaning to the sentences, words like "and", "the" and "in" etc. If we left these words in then there would be a chance that our cosine similarity scores could appear better than they actually are.

In [ ]:
stopwords1 = set(stopwords.words('english'))
gen_res_gru120k_stopwords = []
gen_res_lstm50k_stopwords = []
gen_res_lstm100k_stopwords = []
gen_res_gru100k_stopwords = []
tokenized_targets_stopwords = []

for i in range(len(gru_model_120k)):
  s1stop_removed = [word for word in gru_model_120k[i] if not word.lower() in stopwords1]
  s2stop_removed = [word for word in lstm_model_50k[i] if not word.lower() in stopwords1]
  s3stop_removed = [word for word in lstm_model_100k[i] if not word.lower() in stopwords1]
  s4stop_removed = [word for word in gru_model_100k[i] if not word.lower() in stopwords1]
  s5stop_removed = [word for word in tokenized_targets[i] if not word.lower() in stopwords1]

  s1stop_removed, s2stop_removed, s3stop_removed, s4stop_removed, s5stop_removed = [], [], [], [], []

  for word in gru_model_120k[i]:
    if word not in stopwords1:
        s1stop_removed.append(word)
  gen_res_gru120k_stopwords.append(s1stop_removed)

  for word in lstm_model_50k[i]:
    if word not in stopwords1:
        s2stop_removed.append(word)
  gen_res_lstm50k_stopwords.append(s2stop_removed)

  for word in lstm_model_100k[i]:
    if word not in stopwords1:
        s3stop_removed.append(word)
  gen_res_lstm100k_stopwords.append(s3stop_removed)

  for word in gru_model_100k[i]:
    if word not in stopwords1:
        s4stop_removed.append(word)
  gen_res_gru100k_stopwords.append(s4stop_removed)

  for word in tokenized_targets[i]:
    if word not in stopwords1:
        s5stop_removed.append(word)
  tokenized_targets_stopwords.append(s5stop_removed)



<i>**Word Embedding Issue**</i>

Before conducting the cosine_similarity tests, we first want to note the poor performance of the embeddings that were trained by the models demonstrated in this notebook. The code block below shows how some words which should have good cosine similarity scores do not. We extract the word embeddings for the word’s "doctor" and "patient". Naturally, the vector representations of these words should be close to each other. Yet we are getting a very low score of 0.0373 which does not indicate a very strong similarity between these words.

In [ ]:
doctor_embed = embedding1(torch.LongTensor([voc1.word2index["doctor"]]))
patient_embed = embedding1(torch.LongTensor([voc1.word2index["patient"]]))
result = util.cos_sim(doctor_embed, patient_embed)
print(result)


tensor([[0.0373]], grad_fn=<MmBackward0>)


# **Cosine Similarity Tests**.
We have conducted two sepearte cosine similarity tests below. Due to the issue of a poorly learned word embedding space in this model we have used external models to conduct cosine similarity. The first is the BERT model, where we will compare the cosine similarity of one target sentence from the test_set with generated_responses from the four models tested. The second cosine similarity test simply compares all generated responses from the DocBot for all tested models with their corresponding target sentences.

The BERT model will generate dense embedding vectors and calaculate the cosine for those vectors, as opposed to the embedding vectors created with our model. BERT uses a mean pooling method. It will take the average of all word embeddings to generate a sentence level vector.

This method will generate five sentence embeddings where the first sentence embedding is derived from the i-th target sentence found in the test_set, the other four sentence embeddings represent the corresponding DocBot generated responses. This cosine similarity test simply takes the word embedding vectors of all four DocBot responses and calculates the cosine similarity between them and the target sentence embedding vector. Thus there will be four outputted scores, one for each sentence and we compare them to see which is the highest<sup>[9][10]</sup>.

For emxample, let target sentence = $t_{1}$, let bot response sentences = $b_{1}, b_{2}, b_{3}, b_{4}.$

This cosine similarity test, using sentence vector representation derived from BERT model word embeddings calculates:
> $cosine(t_{1}, b_{1}), cosine(t_{1}, b_{2}), cosine(t_{1}, b_{3}) and cosine(t_{1}, b_{4}).$



<i>**Cosine Similarity Test 1**</i>

Comparing one input from test_set to multiple corresping generated responses from the different trained models to determine which generated_sentences score better.

To be clear, what follows is a comparison of the DocBot generated responses to the actual target sentence. The code snippet below makes this clearer.

In [ ]:
print("THE TARGET SENTENCE FROM TEST_SET:")
print(tokenized_targets_stopwords[0])
print("THE CORRESPONDING BOT RESPONSE FROM GRU MODEL 1:")
print(gen_res_gru120k_stopwords[0])
print("THE CORRESPONDING BOT RESPONSE FROM LSTM MODEL 1:")
print(gen_res_lstm50k_stopwords[0])
print("THE CORRESPONDING BOT RESPONSE FROM LSTM MODEL 2:")
print(gen_res_lstm100k_stopwords[0])
print("THE CORRESPONDING BOT RESPONSE FROM GRU MODEL 2:")
print(gen_res_gru100k_stopwords[0])

THE TARGET SENTENCE FROM TEST_SET:
['hello', '.', 'thank', 'using', 'ask', 'doctor', 'service', '.', 'gone', 'query', 'advice', '.you', 'seem', 'hives', '.', 'hives', 'present', 'itchy', 'welts', 'redness', 'flare', '.', 'allergic', 'reaction', 'food', 'food', 'preservatives', 'common', 'cause', 'hives', '.', 'causes', 'include', 'infection', 'drugs', 'alcohol', 'etc', '.', 'suggest', 'take', 'oral', 'antihistamine', 'e', '.g', 'tab', 'cetrizine', 'mg', 'day', 'days', '.', 'addition', 'suggest', 'apply', 'soothing', 'lotion', 'e', '.g', 'calamine', 'lotion', 'help', 'reduce', 'itching', 'flare', '.hope', 'answered', 'know', 'need', 'assistance', '.']
THE CORRESPONDING BOT RESPONSE FROM GRU MODEL 1:
['hi', 'itchy', 'rash', 'could', 'due', 'contact', 'dermatitis', 'due', 'excessive', 'weather', 'allergy', 'insect', 'bite', 'hypersensitivity', 'reaction', 'proper', 'treatment', 'treated', '.', 'consult', 'dermatologist', 'proper', 'diagnosis', 'treatment', '.', 'hope', 'answered', 'query'

In [ ]:
# COSINE TEST 1

compare_list, compare_list1, compare_list2, compare_list3 = [], [], [], []

# First target sentence compared to four DocBot generated sentences.
compare_list.append(tokenized_targets_stopwords[0])
compare_list.append(gen_res_gru120k_stopwords[0])
compare_list.append(gen_res_lstm50k_stopwords[0])
compare_list.append(gen_res_lstm100k_stopwords[0])
compare_list.append(gen_res_gru100k_stopwords[0])

# Second target sentence compared to four DocBot generated sentences.
compare_list1.append(tokenized_targets_stopwords[1])
compare_list1.append(gen_res_gru120k_stopwords[1])
compare_list1.append(gen_res_lstm50k_stopwords[1])
compare_list1.append(gen_res_lstm100k_stopwords[1])
compare_list1.append(gen_res_gru100k_stopwords[1])

# Third target sentence compared to four DocBot generated sentences.
compare_list2.append(tokenized_targets_stopwords[2])
compare_list2.append(gen_res_gru120k_stopwords[2])
compare_list2.append(gen_res_lstm50k_stopwords[2])
compare_list2.append(gen_res_lstm100k_stopwords[2])
compare_list2.append(gen_res_gru100k_stopwords[2])

# Fourth target sentence compared to four DocBot generated sentences.
compare_list3.append(tokenized_targets_stopwords[3])
compare_list3.append(gen_res_gru120k_stopwords[3])
compare_list3.append(gen_res_lstm50k_stopwords[3])
compare_list3.append(gen_res_lstm100k_stopwords[3])
compare_list3.append(gen_res_gru100k_stopwords[3])

# Get average sentence embeddings (i.e, sentence vector representations)
sentence_embeddings = model.encode(compare_list)
sentence_embeddings1 = model.encode(compare_list1)
sentence_embeddings2 = model.encode(compare_list2)
sentence_embeddings3 = model.encode(compare_list3)

# Calculate cosine similarity for each DocBot response to the corresponding target response.
result = cosine_similarity([sentence_embeddings[0]], sentence_embeddings[1:])
result1 = cosine_similarity([sentence_embeddings1[0]], sentence_embeddings1[1:])
result2 = cosine_similarity([sentence_embeddings2[0]], sentence_embeddings2[1:])
result3 = cosine_similarity([sentence_embeddings3[0]], sentence_embeddings3[1:])

# Display results in a table
table = [["Bot response 1 (GRU-120k)", result[0][0], result1[0][0], result2[0][0], result3[0][0]],
        ["Bot response 2 (LSTM-50k)", result[0][1], result1[0][1], result2[0][1], result3[0][1]],
        ["Bot response 3 (LSTM-100k)", result[0][2], result1[0][2], result2[0][2], result3[0][2]],
        ["Bot response 4 (GRU-100k)", result[0][3], result1[0][3], result2[0][3], result3[0][3]]]

# Display results in a table

column_titles = ["Model", "Score for \n sentence 1", "Score for \nsentence 2",
                 "Score for \n sentence 3", "Score for \n sentence 4"]
print("\033[1m" +"Table 1: Cosine Similarity Results"+ "\033[0m")

print(tabulate(table, headers=column_titles, tablefmt="fancy_grid"))


Table 1: Cosine Similarity Results
╒════════════════════════════╤═══════════════╤══════════════╤═══════════════╤═══════════════╕
│ Model                      │    Score for  │   Score for  │    Score for  │    Score for  │
│                            │    sentence 1 │   sentence 2 │    sentence 3 │    sentence 4 │
╞════════════════════════════╪═══════════════╪══════════════╪═══════════════╪═══════════════╡
│ Bot response 1 (GRU-120k)  │      0.806739 │     0.883814 │      1        │             1 │
├────────────────────────────┼───────────────┼──────────────┼───────────────┼───────────────┤
│ Bot response 2 (LSTM-50k)  │      0.910757 │     0.861588 │      0.987186 │             1 │
├────────────────────────────┼───────────────┼──────────────┼───────────────┼───────────────┤
│ Bot response 3 (LSTM-100k) │      0.915301 │     0.894166 │      0.987186 │             1 │
├────────────────────────────┼───────────────┼──────────────┼───────────────┼───────────────┤
│ Bot response 4 (GRU-100

From the results diplayed above in table 1, it appears that the LSTM models perform slightly better for the first two sentences, the GRU-120k slightly outperforms the other three models and we see an equal performance for sentence 4.

**Cosine Similarity Test 2**


This is an independent binary test that simply outputs one of two predictions, whether the two sentences are similar, or not. The two sentences we compare are the targets from the test_set with the corresponding generated_responses.

This is another way in which we predict whether two sentences are similar to one another. Unlike the previous cosine similarity test, we will process all sentences for our generated test batch (20 sentences in total) compared to the target batch from the test_set.

This test uses the Sentence Transformers API and compares the sentences in a column fashion. For intance, if we have a list of target sentences $T = [t_{1}, t_{2}, ..., t_{n}]$ and a list of bot responses $B = [b_{1}, b_{2}, ..., b_{n}]$, then the transformer method below will calculate the cosine similarity for each corresponding T/B pairs, like so:

> $cosine(T, B) = cosine(t_{1},b_{1}), cosine(t_{2}, b_{2}), ..., cosine(t_{n}, b_{n}).$

Below we will display the results for the first four sentences by using the word "Similar" for a cosine result that is >0.5, and "Not Similar" for a cosine score of <0.5. We will then display a separate table for all sentences calculated, but rather than using the previously mentioned labels we will simply tally how many "Similar" results each model achieved out of the 20 sentences considered<sup>[11]</sup>.


In [ ]:
# COSINE SIMILARITY TEST 2

col1_GRU1 = gen_res_gru120k_stopwords
col2_TAR = tokenized_targets_stopwords

col3_LSTM1 = gen_res_lstm50k_stopwords
col4_TAR = tokenized_targets_stopwords

col5_LSTM2 = gen_res_lstm100k_stopwords
col6_TAR = tokenized_targets_stopwords

col7_GRU2 = gen_res_gru100k_stopwords
col8_TAR = tokenized_targets_stopwords


# Get vector embeddings
vectors1 = model1.encode(col1_GRU1, convert_to_tensor=True)
vectors2 = model1.encode(col2_TAR, convert_to_tensor=True)

vectors3 = model1.encode(col3_LSTM1, convert_to_tensor=True)
vectors4 = model1.encode(col4_TAR, convert_to_tensor=True)

vectors5 = model1.encode(col5_LSTM2, convert_to_tensor=True)
vectors6 = model1.encode(col6_TAR, convert_to_tensor=True)

vectors7 = model1.encode(col7_GRU2, convert_to_tensor=True)
vectors8 = model1.encode(col8_TAR, convert_to_tensor=True)


# Get cosine scores
cosine_scores = util.cos_sim(vectors1, vectors2)
cosine_scores1 = util.cos_sim(vectors3, vectors4)
cosine_scores2 = util.cos_sim(vectors5, vectors6)
cosine_scores3 = util.cos_sim(vectors7, vectors8)

#Display cosine similarity score for the computed embeddings
labels, labels1, labels2, labels3 = [], [], [], []

for i,(sent1,sent2) in enumerate(zip(col1_GRU1,col2_TAR)):
    if cosine_scores[i][i]>=0.5:
      label="Similar"
      labels.append(label)
    else:
      label="Not Similar"
      labels.append(label)

print()

for i,(sent1,sent2) in enumerate(zip(col3_LSTM1,col4_TAR)):
    if cosine_scores1[i][i]>=0.5:
      label="Similar"
      labels1.append(label)
    else:
      label="Not Similar"
      labels1.append(label)

print()

for i,(sent1,sent2) in enumerate(zip(col5_LSTM2,col6_TAR)):
    if cosine_scores2[i][i]>=0.5:
      label="Similar"
      labels2.append(label)
    else:
      label="Not Similar"
      labels2.append(label)

print()

for i,(sent1,sent2) in enumerate(zip(col7_GRU2,col8_TAR)):
    if cosine_scores3[i][i]>=0.5:
      label="Similar"
      labels3.append(label)
    else:
      label="Not Similar"
      labels3.append(label)

# Display results in a table for four sentences using the labels.

table = [["GRU-120k", labels[0], labels[1], labels[2], labels[3]],
        ["LSTM-50k", labels1[0], labels1[1], labels1[2], labels1[3]],
        ["LSTM-100k", labels2[0], labels2[1], labels2[2], labels2[3]],
        ["GRU-100k", labels3[0], labels3[1], labels3[2], labels3[3]]]

column_titles = ["Model", "Sentence Similarity \n sentence 1", "Sentence Similarity \nsentence 2",
                 "Sentence Similarity \n sentence 3", "Sentence Similarity \n sentence 4"]
print("\033[1m" +"Table 2: Results for the first 4 target setneces form the test_set"+ "\033[0m")
print(tabulate(table, headers=column_titles, tablefmt="fancy_grid"))

# Display all results in a table for the 20 sentences processed.

print("\033[1m" +"Table 3: Full results tallied for each model on all 20 test_set sentences"+ "\033[0m")

gru120k_score = 0
lstm50k_score = 0
lstm100k_score = 0
gru100k_score = 0

# tabulate the scores for easier readability, as opposed to create a big table with the labels "Similar and Not Similar"
for i in range(len(labels)):
  if labels[i] == "Similar":
    gru120k_score += 1
  if labels1[i] == "Similar":
    lstm50k_score += 1
  if labels2[i] == "Similar":
    lstm100k_score += 1
  if labels3[i] == "Similar":
    gru100k_score += 1

table1 = [["GRU-120k", gru120k_score],
        ["LSTM-50k", lstm50k_score],
        ["LSTM-100k", lstm100k_score],
        ["GRU-100k", gru100k_score]]

column_titles = ["Model", "Similarity Matches for 20 target responses from test_set"]

print(tabulate(table1, headers=column_titles, tablefmt="fancy_grid"))





Table 2: Results for the first 4 target setneces form the test_set
╒═══════════╤════════════════════════╤════════════════════════╤════════════════════════╤════════════════════════╕
│ Model     │ Sentence Similarity    │ Sentence Similarity    │ Sentence Similarity    │ Sentence Similarity    │
│           │  sentence 1            │ sentence 2             │  sentence 3            │  sentence 4            │
╞═══════════╪════════════════════════╪════════════════════════╪════════════════════════╪════════════════════════╡
│ GRU-120k  │ Not Similar            │ Similar                │ Similar                │ Similar                │
├───────────┼────────────────────────┼────────────────────────┼────────────────────────┼────────────────────────┤
│ LSTM-50k  │ Similar                │ Similar                │ Similar                │ Similar                │
├───────────┼────────────────────────┼────────────────────────┼────────────────────────┼────────────────────────┤
│ LSTM-100k │ Simi

Table 2 shows that the LSTM models are outperforming the GRU models for those four sentences. Perhaps surprisingly, given the lower iterations during the training phase, the LSTM model with just 50,000 iterations and maximum 70 word length obtained more Similarity matches with 15 out of 20. This could be because the lower complexity involved during training as the sequence length was reduced and the attention layer could perform better with a shorter (yet still quite long sentence length). Note this does not necessarily mean it will produce more fluent sentences, which we shall investigate in the qualitative phase of this work in the next section.

##**Readability Tests**

Given that one of our principal goals in this work is to see if the DocBot can produce legible responses we have implemented various readability tests to run the DocBot responses through.

There exists hundreds of different formulas and metrics for measuring the readability of texts<sup>[12]</sup>, too many to conduct here. We utilize the py-readability-metrics suite which consists of 9 different formulas and metrics for assessing the readability of text<sup>[13]</sup>.

For a better comparison, in addition to calculating readability scores for all four models, we also calculated the readability scores for 100 corresponding target sentences for the test set. Given that these sentences were written by humans it should provide for an interesting comparison to our bot responses.

These tests require at least 100 words and they do not take tokenized input. For these large bot response lists and targets list, we have not removed any stopwords or words that are not in the vocabulary as this is not necessary. This is appropriate as this test is not to obtain any semantical meaning in a quantitative way as we attempted using cosine similarity and bleu score.

We will briefly describe the some of the readability functions used and comment on the results.


* flesch_kincaid() - this metric indicates how difficult text is to read. It has been widely adopted to assess the diffiulty in the readiabliity of technical manuals, legal documents and business documents. It is on a scale of 0 - 100<sup>[13]</sup>.

* gunning_fog() - this metric is designed to measure the amount of years of education a person might need to understand the text. "A fog index of 12 requires the reading level of a U.S. high school senior (around 18 years old)"<sup>[13]</sup>.

* dale_chall() - this tests for the familiarity of words in a text. It is based on the insight that it is easier for readers to retain inforamtion if the text contains words that they are familiar with<sup>[13]</sup>.



In [ ]:
gru_model_120k_large = evaluateTestBatch(encoder1, decoder1, searcher1, voc1, large_inputs, 100)
gru_model_100k_large = evaluateTestBatch(encoder4, decoder4, searcher4, voc1, large_inputs, 100)
lstm_model_50k_large = evaluateTestBatch(encoder2, decoder2, searcher2, voc2, large_inputs, 70)
lstm_model_100k_large = evaluateTestBatch(encoder3, decoder3, searcher3, voc1, large_inputs, 100)

This test_batch evaluation is with the GRU RNN
This test_batch evaluation is with the GRU RNN
This test_batch evaluation is with the LSTM RNN
This test_batch evaluation is with the LSTM RNN


In [ ]:
# Convert responses to string.
GRU_120k = GRU_100k = LSTM_50k = LSTM_100k = targets_large  = ""
for i in range(len(gru_model_120k)):
  GRU_120k = GRU_120k + ' '.join(gru_model_120k_large[i])
  GRU_100k = GRU_100k + ' '.join(gru_model_100k_large[i])
  LSTM_50k = LSTM_50k + ' '.join(lstm_model_50k_large[i])
  LSTM_100k = LSTM_100k + ' '.join(lstm_model_100k_large[i])
targets_large = str(large_targets)


In [ ]:
r = Readability(GRU_120k)
r1 = Readability(GRU_100k)
r2 = Readability(LSTM_50k)
r3 = Readability(LSTM_100k)
target = Readability(targets_large)


# store results for GRU1
a1 = r.flesch_kincaid()
a2 = r.flesch()
a3 = r.gunning_fog()
a4 = r.coleman_liau()
a5 = r.dale_chall()
a6 = r.ari()
a7 = r.linsear_write()
a8 = r.smog()
a9 = r.spache()

# store results for GRU2
b1 = r1.flesch_kincaid()
b2 = r1.flesch()
b3 = r1.gunning_fog()
b4 = r1.coleman_liau()
b5 = r1.dale_chall()
b6 = r1.ari()
b7 = r1.linsear_write()
b8 = r1.smog()
b9 = r1.spache()

# store results for LSTM1
c1 = r2.flesch_kincaid()
c2 = r2.flesch()
c3 = r2.gunning_fog()
c4 = r2.coleman_liau()
c5 = r2.dale_chall()
c6 = r2.ari()
c7 = r2.linsear_write()
c8 = r2.smog()
c9 = r2.spache()

# store results for LSTM2
d1 = r3.flesch_kincaid()
d2 = r3.flesch()
d3 = r3.gunning_fog()
d4 = r3.coleman_liau()
d5 = r3.dale_chall()
d6 = r3.ari()
d7 = r3.linsear_write()
d8 = r3.smog()
d9 = r3.spache()

# store results for targets
t1 = target.flesch_kincaid()
t2 = target.flesch()
t3 = target.gunning_fog()
t4 = target.coleman_liau()
t5 = target.dale_chall()
t6 = target.ari()
t7 = target.linsear_write()
t8 = target.smog()
t9 = target.spache()

# Please note, simply for the presentation of this work we have boldened winning scores manually. We ackowledge if this system were tested with new data or new search methods we would have to manually change the bold formatting.
table = [["flesch_kincaid", "{:.4f}".format(a1.score), "{:.4f}".format(b1.score), "{:.4f}".format(c1.score), "{:.4f}".format(d1.score), "\033[1m{:.4f}\033[0m".format(t1.score)],
        ["flesch", "{:.4f}".format(a2.score), "{:.4f}".format(b2.score), "{:.4f}".format(c2.score), "\033[1m{:.4f}\033[0m".format(d2.score), "{:.4f}".format(t2.score)],
        ["gunning_fog","{:.4f}".format(a3.score), "{:.4f}".format(b3.score), "{:.4f}".format(c3.score), "{:.4f}".format(d3.score), "\033[1m{:.4f}\033[0m".format(t3.score)],
        ["coleman_liau", "{:.4f}".format(a4.score), "{:.4f}".format(b4.score), "{:.4f}".format(c4.score), "{:.4f}".format(d4.score), "\033[1m{:.4f}\033[0m".format(t4.score)],
         ["dale_chall", "{:.4f}".format(a5.score), "{:.4f}".format(b5.score), "\033[1m{:.4f}\033[0m".format(c5.score), "{:.4f}".format(d5.score), "{:.4f}".format(t5.score)],
         ["ari", "{:.4f}".format(a6.score), "{:.4f}".format(b6.score), "{:.4f}".format(c6.score), "{:.4f}".format(d6.score), "\033[1m{:.4f}\033[0m".format(t6.score)],
         ["linsear_write", "{:.4f}".format(a7.score), "{:.4f}".format(b7.score), "{:.4f}".format(c7.score), "{:.4f}".format(d7.score), "\033[1m{:.4f}\033[0m".format(t7.score)],
         ["smog", "{:.4f}".format(a8.score), "\033[1m{:.4f}\033[0m".format(b8.score), "{:.4f}".format(c8.score), "{:.4f}".format(d8.score), "{:.4f}".format(t8.score)],
         ["spache", "{:.4f}".format(a9.score), "{:.4f}".format(b9.score), "{:.4f}".format(c9.score), "{:.4f}".format(d9.score), "\033[1m{:.4f}\033[0m".format(t9.score)]]

# Display results in a table

column_titles = ["Score", "GRU1", "GRU2", "LSTM1", "LSTM2", "Targets"]
#"\033[1m" + "DocBot:" + "\033[0m"
print("\033[1m" +"Table 4: Readability Scores"+ "\033[0m" + "(bold text indicates a win)")
print(tabulate(table, headers=column_titles, tablefmt="fancy_grid"))

Table 4: Readability Scores(bold text indicates a win)
╒════════════════╤═════════╤═════════╤═════════╤═════════╤═══════════╕
│ Score          │    GRU1 │    GRU2 │   LSTM1 │   LSTM2 │   Targets │
╞════════════════╪═════════╪═════════╪═════════╪═════════╪═══════════╡
│ flesch_kincaid │  7.5731 │  7.4578 │  7.6775 │  6.466  │    9.0911 │
├────────────────┼─────────┼─────────┼─────────┼─────────┼───────────┤
│ flesch         │ 62.5481 │ 65.5544 │ 59.6684 │ 66.3686 │   56.6728 │
├────────────────┼─────────┼─────────┼─────────┼─────────┼───────────┤
│ gunning_fog    │ 11.0352 │ 10.2984 │ 11.5576 │ 10.0148 │   12.5001 │
├────────────────┼─────────┼─────────┼─────────┼─────────┼───────────┤
│ coleman_liau   │  8.958  │  8.7154 │  9.9318 │  8.2961 │   10.4334 │
├────────────────┼─────────┼─────────┼─────────┼─────────┼───────────┤
│ dale_chall     │  8.8124 │  9.3628 │  9.7158 │  9.3373 │    9.5039 │
├────────────────┼─────────┼─────────┼─────────┼─────────┼───────────┤
│ ari            │  6.

From the results displayed in table 4 we see that although the scores are low, if we look at the target sentence scores in the final column, we can see that the bot response scores are not far off from scores generated by calculating sentences written by humans, in fact the bot responses sometimes beat the human responses, for example in the dale_chall() and smog() tests. As noted in the pre-processing stage of this work, some of the transcripts used to train the model were automated from speech to text, this could explain why our DocBot is producing sentences that might be as coherent as human generated text. As we have repeated throughout this notebook though, we implore the reader to hold off from interpreting these results as positive until the next section.

# **Qualitative Evaluation.**

In this section we shall demonstrate talking to the DocBot and testing it with query's that the author has constructed. These queries were informed by illness and symptom information extracted from sources such as the National Health Service (NHS) and WebMD. To be clear, this section does not test the DocBot with any sentences from the datasets that were used to train or test the DocBot. This phase of the evaluation is completely up to the user (the author in this case).
The qualitative evaluation will be broken down into three phases. For the first part of this evaluation, we are simply going to talk to the chatbot using the first five lines of our nhs set and discuss what the illnesses were according to the NHS site and look to see what the different versions of DocBot produced as a response. Again, we will be asking two primary questions, were the responses legible and were they at all relevant?

It  is an interesting exercise to simply talk to the DocBot about health-related matters on the fly as the DocBot will sometimes produce legible responses and relevant substrings. For example, we input a query about a real health problem the author is currently experiencing.

For the second phase we will conduct some unstructured interactions with the DocBot. We will test its capacity to respond to inputs of a mental health nature as these types of queries can be very vague in terms of what is wrong with the patient.

For the third phase of this evaluation will shall run a readability test on the entire NHS batch to see if we can tie this part of the evaluation in with the quantitative part of the evaluation. Here we are checking to see whether the LSTM is performing better given that the quantitative evaluation suggests it might, i.e, is it producing more readable human like sentences and are they relevant to the input queries.



First we will import our nhs.txt file and save it into a list so that we can process it later as we did for the test_set.

In [ ]:
nhs_file_path = os.path.join(corpus, "nhs.txt")
nhs_input_queries = []
nhs_file = open(nhs_file_path, 'r')
for line in nhs_file:
  nhs_input_queries.append(line.strip())

Below are the inputs that we derivided from the NHS website.

In [ ]:
for i in range(20):
  print(i+1, nhs_input_queries[i])

1 hello doctor, I have a blocked or runny nose a sore throat, headaches, muscle aches, a cough, I'm sneezing, I have a raised temperature pressure in your ears and face loss of taste and smell
2 hello doctor, i get frequent headaches, muscle aches, a cough and rapid heartbeat i get chest pain which gets worse when breathing or coughing
3 hello doctor i have widespread pain Extreme sensitivity Stiffness Fatigue Poor sleep quality Cognitive problems Headaches Irritable bowel syndrome
4 hi doctor, my eyes are red, they are burning and itchy, they produce pus that sticks to lashes and they are very watery
5 hi doctor, I have a problem, my skin or the whites of my eyes turn yellow
6 hello doctor, I have sore throat, headache, runny or blocked nose, aches and pains, tiredness
7 hi doctor, I feel restlessness, I have a sense of dread, feeling constantly on edge, difficulty concentrating, irritability and angry
8 hello doctor I have a fear of ‘being fat’ or gaining weight, I have problems with

Here are the corresponding diagnoses that the above symptoms suggest according to the NHS website:


Line Number  | Diagnosis
-------------------|------------------
1       | Common Cold
2       | Pnemonia
3       | Fibromyalgia
4       | Conjunctivitis
5       | Jaundice
6       | Bronchitis
7       | Anxiety
8       | Anorexia
9       | Acne
10       | Chicken Pox
11      | Irritable Bowel Syndrome
12       | Huntingtons Disease
13       | Insomnia
14       | Parkinsons Disease
15       | Shingles   
16       | Type-2 Diabetes  
17       | Epilepsy
18       | Conjuctivitis
19       | Asthma
20       | Melanoma


Now we shall run our nhs input queries through each model. We will use these lists for the third phase of this qualitative evaluation.

In [ ]:
nhs_bot_responses_GRU1 = evaluateTestBatch(encoder1, decoder1, searcher1, voc1, nhs_input_queries, 100)
nhs_bot_responses_LSTM1 = evaluateTestBatch(encoder2, decoder2, searcher2, voc2, nhs_input_queries, 70)
nhs_bot_responses_LSTM2 = evaluateTestBatch(encoder3, decoder3, searcher3, voc1, nhs_input_queries, 100)
nhs_bot_responses_GRU2 = evaluateTestBatch(encoder4, decoder4, searcher4, voc1, nhs_input_queries, 100)

This test_batch evaluation is with the GRU RNN
This test_batch evaluation is with the LSTM RNN
This test_batch evaluation is with the LSTM RNN
This test_batch evaluation is with the GRU RNN


### **Phase 1: First five responses to the NHS dataset.**

After inputting five questions to the DocBot, we will make some brief observations about some of the responses before moving onto the next model.

In [ ]:
# First five inputs with the first GRU model
evaluateInput(encoder1, decoder1, searcher1, voc1, 100)

This evaluation is with the GRU RNN
> hello doctor, I have a blocked or runny nose a sore throat, headaches, muscle aches, a cough, I'm sneezing, I have a raised temperature pressure in your ears and face loss of taste and smell
DocBot:hello . take more rest and an appropriate medication . you will include a decongestant like levocetrizine mg
daily i think an allergic reaction . hope it helps . consult a general physician for physical examination . if the count is
already repeated i suggest you to start an anti inflammatory drug like levocetrizine or pregabalin for local application .
this will resolve the problem . nasal block drops at night . for more information consult a family physician online https www
.icliniq .com ask a doctor online family physician medicine . a nasal medicine

> hello doctor, i get frequent headaches, muscle aches, a cough and rapid heartbeat i get chest pain which gets worse when breathing or coughing
DocBot:hello . i went through the post endoscopy . for mo

The DocBot struggles with personal pronouns. To take an example from the first line of the first response, "you will include a decongestant like levocetrizine". This does not make sense. For the first question which are symptoms for a common cold, we note that the DocBot has somewhat conveniently recommended the patient take "appropriate medication". Given the scale of this project and its modesty, we do not see this type of response as a negative. In the same way we did not remove URL's to legit medical sources, if the DocBot simply advises the patient to see an actual doctor, or visit a more appropriate source, this we consider to be good advice.
The final question which are symptoms for jaundice produces a borderline legible response, but a certain response that is not correct.


In [ ]:
# First five inputs with the second GRU model
evaluateInput(encoder4, decoder4, searcher4, voc1, 100)

This evaluation is with the GRU RNN
> hello doctor, I have a blocked or runny nose a sore throat, headaches, muscle aches, a cough, I'm sneezing, I have a raised temperature pressure in your ears and face loss of taste and smell
DocBot:hi i understand your concern . i would like to inform you that your blood count are a benign disorder . it is
a common problem in the tonsils and neck . it is a common problem in the airway . it is a benign condition . i would suggest
you to get an ultrasound to look for the cervical spine and aorta . hope i have answered your query . let me know if i can
assist you further . regards dr . bhanu partap message me a specialist

> hello doctor, i get frequent headaches, muscle aches, a cough and rapid heartbeat i get chest pain which gets worse when breathing or coughing
DocBot:yeah i think its a very high grade musculoskeletal disorder . for more information consult a general physician
online https www .icliniq .com ask a doctor online general health querr

The DocBot does assume that it was give information from the input sequence that it was not. For instance, the first query response the DocBot states it has seen the patients’ blood count. The second GRU model produces an interesting response to query number 5. The DocBot states that the patient should "consult a nearby dermatologist". This is interesting because jaundice does produce a yellowing of the skin.

In summary, the second GRU model with reduced complexity during the training phase thanks to a 200 hidden size and only 1 layer for the encoder and decoder produces somewhat readable substrings comparable to the previous GRU version. Of note are the beginnings of the final three responses:


> "yeah i think its a very high grade musculoskeletal disorder . for more information consult a general physician
online https www .icliniq .com"

> "hi thanks for your query .apply steroid such as clotrimazole cream twice daily"

> "hello and welcome to ask a doctor service . i have reviewed your query and here is my advice"

In [ ]:
# First five inputs with the first LSTM model
evaluateInput(encoder2, decoder2, searcher2, voc2, 70)

This evaluation is with the LSTM RNN
> hello doctor, I have a blocked or runny nose a sore throat, headaches, muscle aches, a cough, I'm sneezing, I have a raised temperature pressure in your ears and face loss of taste and smell
DocBot:hi . i have gone through the complaints . for further queries consult an ent otolaryngologist online https www
.icliniq .com ask a doctor online ent otolaryngologist asthma specialist and traumatologist . asthma and traumatologist
swelling or mouth areas hot mouth spicy food mouth etc . mouth mouth etc a clean pack . n mouth mouth can clean

> hello doctor, i get frequent headaches, muscle aches, a cough and rapid heartbeat i get chest pain which gets worse when breathing or coughing
DocBot:hello . for more information consult an internal medicine physician online https www .icliniq .com ask a
doctor online internal medicine physician medicine specialist medicine and traumatologist counsellor eye counsellor can be
added food can be helpful upper meals .

The LSTM model trained on 70 words length with 50,000 iterations of training struggles to produce any relevant or fluent sentences. Some of the beggining substrings are nearly fluent.

In [ ]:
# First five inputs with the second LSTM model
evaluateInput(encoder3, decoder3, searcher3, voc1, 100)

This evaluation is with the LSTM RNN
> hello doctor, I have a blocked or runny nose a sore throat, headaches, muscle aches, a cough, I'm sneezing, I have a raised temperature pressure in your ears and face loss of taste and smell
DocBot:hello . ent problems are a common cause for a sinus infection especially if the nose is not associated with a
headache or not . this condition may be a result of a viral infection but a headache may be secondary to prolonged relief
with a headache or throat infection . this may be a concern for a few days or may be an infection . for further information
consult a dentist online https www .icliniq .com ask a doctor online dentist a long physician the trigger of

> hello doctor, i get frequent headaches, muscle aches, a cough and rapid heartbeat i get chest pain which gets worse when breathing or coughing
DocBot:hi . it looks like a muscle weakness . you need to take some muscle cough and avoid medications for relief .
also take some multivitamins like ta

The LSTM version trained with 100,000 iterations shows some promise. The first reply wrongly diagnoses the Common Cold, but does suggest an Ear Nose and Throat diagnosis and produces a fairly coherent and accurate substring, but not relevant to the input query:

> hello . ent problems are a common cause for a sinus infection

However, in the same reply the DocBot reccomends seeing a dentist.

Again, the DocBot produces a lot of incoherant substrings.

*Finally*, out of curiosity the author queried the DocBot with a mild medical issue he is currently having with his knee. This is undiagnosed, physio therapists the author has seen suggest possible knee patellar tracking disorder or sciatica.  


In [ ]:
# first GRU model
evaluateInput(encoder1, decoder1, searcher1, voc1, 100)

This evaluation is with the GRU RNN
> Hi doctor, I am concerned about an electric shock like pain in my left inner knee. It happens either when I'm walking or when sitting still sometimes, I can still exercise and run, which is good. The pain is short and sharp, it is gone by the time I notice it.
DocBot:hi . for further information consult a general surgeon online https www .icliniq .com ask a doctor online
general surgeon . motion of the knee got it for days motion of pain . motion of the spine . motion of motion and diabetes .
motion of pain in thigh . motion of motion and delivery . motion of the following . motion of those motion numbness fast
inflammation motion of the spine . motion of the following . motion of those motion numbness fast tissues fast

> q


In [ ]:
# second GRU model
evaluateInput(encoder4, decoder4, searcher4, voc1, 100)

This evaluation is with the GRU RNN
> Hi doctor, I am concerned about an electric shock like pain in my left inner knee. It happens either when I'm walking or when sitting still sometimes, I can still exercise and run, which is good. The pain is short and sharp, it is gone by the time I notice it.
DocBot:hi . revert back with the answers to the above questions to a neurologist online https www .icliniq .com ask a
doctor online neurologist you a online https www .icliniq .com ask a doctor online physiotherapist you a questions you a
happy to help you . you may be happy to help you . you may be happy to help you . you may discuss this further questions you
may want to ask me .stay healthy . ahead you can see a doctor online https www

> q


In [ ]:
# first LSTM model
evaluateInput(encoder2, decoder2, searcher2, voc2, 70)

This evaluation is with the LSTM RNN
> Hi doctor, I am concerned about an electric shock like pain in my left inner knee. It happens either when I'm walking or when sitting still sometimes, I can still exercise and run, which is good. The pain is short and sharp, it is gone by the time I notice it.
DocBot:hi . i understand your concern . for further information consult an internal medicine physician online https
www .icliniq .com ask a doctor online internal medicine physician https www .icliniq .com ask a doctor online internal
medicine physician https www .icliniq .com ask a doctor online internal medicine physician https www .icliniq .com ask a
doctor online internal medicine

> q


In [ ]:
# second LSTM model
evaluateInput(encoder3, decoder3, searcher3, voc1, 100)

This evaluation is with the LSTM RNN
> Hi doctor, I am concerned about an electric shock like pain in my left inner knee. It happens either when I'm walking or when sitting still sometimes, I can still exercise and run, which is good. The pain is short and sharp, it is gone by the time I notice it.
DocBot:hello . for further information consult an orthopaedician and traumatologist online https www .icliniq .com
ask a doctor online orthopaedician and traumatologist a joint . for almost one . a knee joint muscle pain . at that time .
pain in the neck region like neck and breathing difficulty . like pulpitis . . . . . . . . . . . . . . . . .

> q


The final LSTM model that was trained on 100 words correctly recommended an orthopaedist, a specialist surgeon concerned with muscular skeletal conditions. Although the spelling is incorrect. The response from this version of the bot also included the phrase "a knee joint muscle pain", which is highly relevant. Again, as seen throughout this study, relevant words and phrases are surrounded by unintelligible phrases.

**Phase 2: Unstructured Interaction focused on Mental Health Ailemnts.**

In this part of the evaluation, we will interact with the DocBot with no predefined questions. This section concerns how the DocBot responds to random query's. The author is essentially taking on the role of a random patient or user of the system with complaints that suggest mental health problems. These can be very vague and difficult to diagnose.

Dataset 2 was primarily made up of mental health related queries, but it was quite small compared to dataset 4. We will test to see if the DocBot can recognise this type of query. The author came up with some queries that might be mental health related. Due to the fact we are not a doctor or psychiatrist, we found it difficult to come up with mental health related queries.

> Add blockquote



To aid in this we searched the internet for some comments from users on forums that discussed ailments like depression and anxiety, we copy and pasted them into the DocBot prompt, after slightly altering the comments by appending the words "hi" or "hello" "doctor" and changing personal pronouns from "you" to "I". For instance, the following comment was made on Reddit to a post asking "How do you know you are depressed?", the top comment reads:

> "Being emotionally numb. Nothing makes you cry. Nothing makes you laugh. Isolation. Exhaustion. Life on autopilot. Almost a feeling of physical decay. Like your body is slowing down."

Before inputting this into the DocBot prompt we changed the personal pronouns and added the phrase "hi doctor", like so:

> "Hi doctor, I am emotionally numb. Nothing makes me cry. Nothing makes me laugh. I feel isolated. Exhausted. My Life is on autopilot. Almost a feeling of physical decay. Like my body is slowing down."

We also took user comments from posts entitled "People with anxiety, what is anxiety really like?".
Using online forums we constructed five mental health related queries to test the DocBot, we will list them below and then input them into our four models to analyse the responses.
1.	hello doctor, i feel sad, anxious, depressed and worried, i have suicidal thoughts
2.	hi doctor, I am emotionally numb. Nothing makes me cry. Nothing makes me laugh. I feel isolated. Exhausted. My Life is on
3.	hello doctor, I don't wanna do anything, I feel like crying for no reason, I can't eat, I can't sleep all too well, I stay away from everyone, I have a voice in my head that just keeps repeating every thing bad about my life.
4.	hello doctor, I feel like I'm in a constant fog state. I have trouble listening. I know what I should say but I can't say it. My body and everything in it is just frozen.
5.	Hi doctor, I just can't stop thinking, thinking of all the possible conversations I could have and the responses, or all the conversations I have had that I could have given a better response. I always replay anxious situations over and over in my head. I wonder if the path I chose was wrong. I profusely sweat when talking to people. I also procrastinate to take my mind off of anything and everything. autopilot. Almost a feeling of physical decay. Like my body is slowing down.
Importantly, in this section we wanted to keep out any words that are associated with physiological problems (referencing pain in specific areas of the body, blood and organs etc). In essence, we wanted to keep these queries quite vague.



In [ ]:
# First five inputs with the first GRU model
evaluateInput(encoder1, decoder1, searcher1, voc1, 100)

This evaluation is with the GRU RNN
> hello doctor, i feel sad, anxious, depressed and worried, i have suicidal thoughts
DocBot:hi . i have seen your query . just along with yoga and vegetables along with diet rich in milk and do not feel
better . however i would like to know your age in more information revert back with the asked details and a detailed history
to a psychiatrist online https www .icliniq .com ask a doctor online psychiatrist and write the time .

> hi doctor, I am emotionally numb. Nothing makes me cry. Nothing makes me laugh. I feel isolated. Exhausted.  
DocBot:hello . i read your query . for further information consult a psychiatrist online https www .icliniq .com ask
a doctor online psychiatrist

> hello doctor, I don't wanna do anything, I feel like crying for no reason, I can't eat, I can't sleep all too well, I stay away from everyone, I have a voice in my head that just keeps repeating every thing bad about my life.
DocBot:hello . yes it is a panic problem and 

Perhaps the best response seen so far comes from the first GRU trained model, where to the second input query we see it produce a short fluent and relevant reply:

> "hello . i read your query . for further information consult a psychiatrist online https www .icliniq .com ask
a doctor online psychiatrist"

In [ ]:
# Five mental health related queries with the second GRU model
evaluateInput(encoder4, decoder4, searcher4, voc1, 100)

This evaluation is with the GRU RNN
> hello doctor, i feel sad, anxious, depressed and worried, i have suicidal thoughts
DocBot:hello and welcome to ask a doctor service . i have reviewed your query and here is my advice . i have read
your message and understand your concerns . i hope you are satisfied with my answer . feel free to write back to me if you
have any other queries i will be happy to help you further . wishing you good health . regards happy to help you psychiatrist
a specialist psychiatrist message me .com a direct dd ae d ec ecdd congenital cataracts message me . to help

> hi doctor, I am emotionally numb. Nothing makes me cry. Nothing makes me laugh. I feel isolated. Exhausted. My Life is on
DocBot:hello . i read carefully through your question and understand your concerns . for further information consult
an internal medicine physician online https www .icliniq .com ask a doctor online internal medicine physician specialist the
help of management and aorta . you are d

In [ ]:
# Five mental health related queries with the first LSTM model
evaluateInput(encoder2, decoder2, searcher2, voc2, 70)

This evaluation is with the LSTM RNN
> hello doctor, i feel sad, anxious, depressed and worried, i have suicidal thoughts
DocBot:hello . from how much it can be so if you had your symptoms forward so that it is less in your last period .
the exact cause are of so many side effects is to be avoided . iron and calcium intake of foods is also a very good way . you
can get it done . to see how with the episode of help .

> hi doctor, I am emotionally numb. Nothing makes me cry. Nothing makes me laugh. I feel isolated. Exhausted. My Life is on
DocBot:hi . for further information consult a psychiatrist online https www .icliniq .com ask a doctor online
psychiatrist asthma specialist medicine specialist and https www .icliniq .com ask a doctor online ayurveda specialist
medicine specialist r food and can i can assist you further . asthma and other doctor with medicines https www .icliniq .com
ask

> hello doctor, I don't wanna do anything, I feel like crying for no reason, I can't eat, I can'

It is interesting to note that the DocBot again suggests seeing a psychiatrist for some of these mental health related queries. To our knowledge, the DocBot made no such recommendations to the classic medical problems previously inputted to the DocBot.

In [ ]:
# First five inputs with the second LSTM model
evaluateInput(encoder3, decoder3, searcher3, voc1, 100)

This evaluation is with the LSTM RNN
> hello doctor, i feel sad, anxious, depressed and worried, i have suicidal thoughts
DocBot:hello . i hope this helps you to ask a doctor service . get a good approach to an ayurveda specialist and get
a prescription for the right treatment . mandatory . in ayurveda time for the duration to five weeks . if you have any
further questions please do not hesitate to contact me . you are writing to ask any further queries . you are welcome . you
should be good . good luck . https www .icliniq .com ask a doctor online neurologist good health to

> hi doctor, I am emotionally numb. Nothing makes me cry. Nothing makes me laugh. I feel isolated. Exhausted. My Life is on
DocBot:hello . thanks for writing to us . the symptoms that you have mentioned are not related to a possible major
damage . a thing is possible to explain the problem of a major disorder . a genetic disorder . a complete blood count . i
would suggest you to get a careful thyroid profile blood

**Short sentences Issue**.

We noticed that the DocBot does not handle short sentences very well. We suspect that this is due to the infrequent short sentences that were available in the data sets. In this section we will show how the DocBot responds to short sentences.

In [ ]:
evaluateInput(encoder1, decoder1, searcher1, voc1, 100)


This evaluation is with the GRU RNN
> Hi doctor, I'm sick
DocBot:hi . this is not like a long term . but do you ever like eating something or something anxious and then you
can go back to the gym . for more information consult a sexologist online https www .icliniq .com ask a doctor online
sexologist

> hello!
DocBot:i would under my doctor . . . but i dont know . im not sure why is it . on the face and points me up . its
like im a weird like if i didnt really try looking up . on the face a lot . not going to say that if i didnt get it removed
before . not its like oh it would be that if something is going to be a deep connection . i would just like not . on the face
. not got it .

> I can't sleep
DocBot:hi i think there is some one thing that one of the things are less . that was just one thing . i guess you
could take some extra medicines for it . but i dont think i do . i dont know . if its not very much anyway then you could
take another different options . that will give you bett

We also observe that the DocBot will give different answers depending on whether we start the sentence with either "hello doctor", or "hi docotor". The second set of queries below shows that the DocBot gives a more confusing response if we don't use the "hi doctor" substring at the start of the query.

In [ ]:
evaluateInput(encoder1, decoder1, searcher1, voc1, 100)

This evaluation is with the GRU RNN
> i ate something an i developed rash on my body . nothing serious just really small red dots and bit itchy . i don t have any other food allergy or food poisoning symptoms . . .it happened before after i ate meat medium beef for ex . what can i use to apply on my skin ? thank you
DocBot:hi the rash could be an allergic reaction to your history . insect bite occurs due to insect bite
hypersensitivity reaction or environmental disease conditions you might require treatment with calamine lotion for few days .
apply calamine lotion and take an antihistamine like levocetirizine for one to four days . if your allergic reaction develops
during contact with itching and consult your dermatologist .hope i have answered your query . let me know if i can assist you
further . regards dr . albana sejdini general family physician to a dermatologist and

> hi doctor, i ate something an i developed rash on my body . nothing serious just really small red dots and bit

**Phase 3: Readability Tests**

As we cannot perform any meaningful cosine or BLEU tests given that we do not have any comparison sentences for our NHS test set, we conducted another readiability test as we did during the quantative stage for the test_set.

In [ ]:
# Convert responses to string.
GRU1 = GRU2 = LSTM1 = LSTM2 = ""
for i in range(len(nhs_bot_responses_GRU1)):
  GRU1 = GRU1 + ' '.join(nhs_bot_responses_GRU1[i])
  GRU2 = GRU2 + ' '.join(nhs_bot_responses_GRU2[i])
  LSTM1 = LSTM1 + ' '.join(nhs_bot_responses_LSTM1[i])
  LSTM2 = LSTM2 + ' '.join(nhs_bot_responses_LSTM2[i])

In [ ]:
# initialize Readability objects for each model
r = Readability(GRU1)
r1 = Readability(GRU2)
r2 = Readability(LSTM1)
r3 = Readability(LSTM2)

# store results for GRU1
a1 = r.flesch_kincaid()
a2 = r.flesch()
a3 = r.gunning_fog()
a4 = r.coleman_liau()
a5 = r.dale_chall()
a6 = r.ari()
a7 = r.linsear_write()
a8 = r.smog()
a9 = r.spache()

# store results for GRU2
b1 = r1.flesch_kincaid()
b2 = r1.flesch()
b3 = r1.gunning_fog()
b4 = r1.coleman_liau()
b5 = r1.dale_chall()
b6 = r1.ari()
b7 = r1.linsear_write()
b8 = r1.smog()
b9 = r1.spache()

# store results for LSTM1
c1 = r2.flesch_kincaid()
c2 = r2.flesch()
c3 = r2.gunning_fog()
c4 = r2.coleman_liau()
c5 = r2.dale_chall()
c6 = r2.ari()
c7 = r2.linsear_write()
c8 = r2.smog()
c9 = r2.spache()

# store results for LSTM2
d1 = r3.flesch_kincaid()
d2 = r3.flesch()
d3 = r3.gunning_fog()
d4 = r3.coleman_liau()
d5 = r3.dale_chall()
d6 = r3.ari()
d7 = r3.linsear_write()
d8 = r3.smog()
d9 = r3.spache()

# Display results in a table

table = [["flesch_kincaid", "{:.4f}".format(a1.score), "{:.4f}".format(b1.score), "\033[1m{:.4f}\033[0m".format(c1.score), "{:.4f}".format(d1.score)],
        ["flesch", "{:.4f}".format(a2.score), "{:.4f}".format(b2.score), "{:.4f}".format(c2.score), "\033[1m{:.4f}\033[0m".format(d2.score)],
        ["gunning_fog","{:.4f}".format(a3.score), "{:.4f}".format(b3.score), "\033[1m{:.4f}\033[0m".format(c3.score), "{:.4f}".format(d3.score)],
        ["coleman_liau", "{:.4f}".format(a4.score), "{:.4f}".format(b4.score), "\033[1m{:.4f}\033[0m".format(c4.score), "{:.4f}".format(d4.score)],
         ["dale_chall", "{:.4f}".format(a5.score), "{:.4f}".format(b5.score), "\033[1m{:.4f}\033[0m".format(c5.score), "{:.4f}".format(d5.score)],
         ["ari", "{:.4f}".format(a6.score), "{:.4f}".format(b6.score), "\033[1m{:.4f}\033[0m".format(c6.score), "{:.4f}".format(d6.score)],
         ["linsear_write", "{:.4f}".format(a7.score), "{:.4f}".format(b7.score), "\033[1m{:.4f}\033[0m".format(c7.score), "{:.4f}".format(d7.score)],
         ["smog", "{:.4f}".format(a8.score), "{:.4f}".format(b8.score), "\033[1m{:.4f}\033[0m".format(c8.score), "{:.4f}".format(d8.score)],
         ["spache", "{:.4f}".format(a9.score), "{:.4f}".format(b9.score), "\033[1m{:.4f}\033[0m".format(c9.score), "{:.4f}".format(d9.score)]]

column_titles = ["Test", "GRU1", "GRU2", "LSTM1", "LSTM2"]
print("\033[1m" +"Table 4: Readability Scores"+ "\033[0m " + "(bold text indicates a win)")
print(tabulate(table, headers=column_titles, tablefmt="fancy_grid"))

Table 4: Readability Scores (bold text indicates a win)
╒════════════════╤═════════╤═════════╤═════════╤═════════╕
│ Test           │    GRU1 │    GRU2 │   LSTM1 │   LSTM2 │
╞════════════════╪═════════╪═════════╪═════════╪═════════╡
│ flesch_kincaid │  7.3413 │  6.5028 │  8.2691 │  6.1887 │
├────────────────┼─────────┼─────────┼─────────┼─────────┤
│ flesch         │ 61.894  │ 67.9921 │ 55.7832 │ 68.2998 │
├────────────────┼─────────┼─────────┼─────────┼─────────┤
│ gunning_fog    │ 11.249  │  9.5537 │ 11.5921 │  8.9744 │
├────────────────┼─────────┼─────────┼─────────┼─────────┤
│ coleman_liau   │  9.4513 │  7.9489 │  9.8617 │  7.8326 │
├────────────────┼─────────┼─────────┼─────────┼─────────┤
│ dale_chall     │  9.2509 │  8.3742 │  9.851  │  8.2673 │
├────────────────┼─────────┼─────────┼─────────┼─────────┤
│ ari            │  6.5725 │  5.3851 │  7.0031 │  4.9522 │
├────────────────┼─────────┼─────────┼─────────┼─────────┤
│ linsear_write  │  6.5991 │  6.1424 │  6.8736 │  5.4    │


Table 4 results show that the LSTM models outperform the GRU models, specifically, LSTM 1 wins all but one readability test. This LSTM model was trained with fewer words and only 50,000 iterations, it had the lowest average loss of all four models.

# **Conclusion**

Overall, we are slightly disappointed with the performance of the DocBot, but given some of the responses observed with occasional relevant substrings and a lot of fluent sentences (even if they were not relevant), we feel this work could be a good starting point for further investigation and testing to develop a similar system with greater scale in terms of more training iterations and more diverse datasets that contain shorter sentences and non-medical text.

A few items to note in conclusion is that the average loss on the range that was exhibited during training for the four models presented in this notebook did not seem to correlate with better quality output from the bot. The BLEU, cosine similarity and readability tests did not seem to correlate with the performance of the DocBot, or at the very least, given the poor scores observed in the quantitative test the impact on what the bot produced was negligible.

It is interesting to note that even the LSTM model that was trained with 50,000 iterations as opposed to both GRU models which were trained with at least 100,0000 iterations, managed to slightly outperform the GRU models during cosine similarity testing. We found that lowering the complexity of the model by reducing the number of layers and the hidden size did not negatively affect the model's performance compared to the default 500 hidden size and 2 layers for each encoder and decoder.





# **Future Recommendations.**

* Larger dataset that contains more shorter sentences. We would combine a larger non-medical corpus.
* More training. 120,000 iterations took over 24 hours to complete. Yet we would seek to complete more iterations to see if lower average loss could aid in more readable and relevant responses.
* Expert insight to better evaluate the DocBot.

# **References**

[1] Patek, Pranay (2020) Disease Symptom Prediction Dataset
 Dataset [Available from] https://www.kaggle.com/datasets/itachi9604/disease-symptom-description-dataset?select=dataset.csv

[2] Nazmul, K; et al (2020) dataset_automated_medical_transcription
 Dataset [Available from] https://github.com/nazmulkazi/dataset_automated_medical_transcription

[3] Fareez, F; et al (2022) A dataset of simulated patient-physician medical interviews with a focus on respiratory cases
 Article [Available from] https://www.nature.com/articles/s41597-022-01423-1#Sec5

[4] Fareez, F; et al (2022) A dataset of simulated patient-physician medical interviews with a focus on respiratory cases
 Dataset [Available from] https://figshare.com/s/d83162fad67407081b32

[5] Ahmed; et al (2022) Dianose Me.
 Dataset [Available from] https://www.kaggle.com/datasets/dsxavier/diagnoise-me

[6] Inkawhich, M. Chatbot Tutorial
 Notebook [Available from] https://pytorch.org/tutorials/beginner/chatbot_tutorial.html

[7] Brownlee, J (2002). A Gentle Introduction to Calculating the BLEU Score for Text in Python [Available from] https://machinelearningmastery.com/calculate-bleu-score-for-text-python/

[8] Kishore Papineni, Salim Roukos, Todd Ward, and Wei-Jing Zhu (2002). BLEU: a Method for Automatic Evaluation of Machine Translation

[9] Walia, S, M (2021). Measuring Text Similarity Using BERT. Article. [Available from] https://www.analyticsvidhya.com/blog/2021/05/measuring-text-similarity-using-bert/

[10] Briggs, J (2021). BERT For Measuring Text Similarity
. Article. [Available from] https://towardsdatascience.com/bert-for-measuring-text-similarity-eec91c6bf9e1

[11] Abraham, N, A (2021). Sentence Embeddings with PyTorch Lightning. Article. [Available from] https://blog.paperspace.com/sentence-embeddings-pytorch-lightning/

[12] Bellot, P; Tavernier, J (2012). Flesch and Dale-Chall Readability Measures for INEX 2011 Question-Answering Track. Conference Paper. [Available from] https://www.researchgate.net/profile/Patrice-Bellot/publication/258099000_Flesch_and_Dale-Chall_Readability_Measures_for_INEX_2011_Question-Answering_Track/links/00b49526eb7e6e1983000000/Flesch-and-Dale-Chall-Readability-Measures-for-INEX-2011-Question-Answering-Track.pdf

[13] DiMascio, M, Carmine (2018) py-readability-metrics source code (Version 1.4.4) [Source code]. https://github.com/cdimascio/py-readability-metrics

[14] Vinyals, Q;, Quoc, Le (2015). A Neural Conversational Model. Article. [Available from] https://arxiv.org/abs/1506.05869

[15] Luong, Minh-Thang; et al (2015). Effective Approaches to Attention-based Neural Machine Translation. Article. [Available from] https://arxiv.org/abs/1508.04025

[16] Ng, Andrew. (2019, August 28). Sequence Models Complete Course. [Video]. YouTube. https://www.youtube.com/watch?v=S7oA5C43Rbc&ab_channel=ExploreTheKnowledge


